# Notebook 2: Advanced RAG Techniques & LLM-as-Judge Evaluation

**Prerequisites:** NB3 (Mistral optimization) completed. Best config: chunk1500, k=8, similarity, mpnet, recursive.

### Baseline
- **NB3 best (Mistral):** chunk1500_o300_k8 — composite **0.6111** (8q), **0.5700** (15q expanded)
- **First Cohere run:** chunk1000_200_mpnet — composite **0.5380** (full 75q)
- **Known weaknesses:**
  - Q13 (Longformer): composite **0.29-0.37** — retrieval failure
  - Engineering persona underperforms Marketing by ~0.017
  - Evaluation/hallucination question clusters (40% of questions) untested until now

### What this notebook does
1. **HyDE query expansion** — hypothetical document embeddings
2. **Parent Document Retrieval** — small chunks for retrieval, large for generation
3. **Cross-Encoder Reranker** — retrieve broad, rerank to top-k
4. **Technique combinations** — HyDE+Reranker, Parent+Reranker (Mistral, 0 Cohere calls)
5. **G-Eval dry run with Mistral** — validate judge prompt before spending Cohere calls
6. **G-Eval with Cohere** — reference-based scoring 1-5
7. **Full 75-question final run** with best technique + Cohere

### Subset strategy
10-question subset (improved from 8q after topic cluster audit). Added Q102 (evaluation) and Q212 (hallucination) to fill the two biggest coverage gaps. See EXPERIMENT_LOG.md § 2 for full rationale.
Subset scores are ~7% optimistically biased vs full run — consistent across experiments.

## 1. Setup

In [1]:
%%capture
!pip install langchain langchain-community langchain-huggingface langchain-qdrant langchain-cohere
!pip install langchain-classic
!pip install qdrant-client pymupdf wikipedia bs4 sentence-transformers transformers
!pip install bitsandbytes accelerate rouge_score
!pip install -U langchain-text-splitters
!pip install sentence-transformers[cross-encoder]

In [2]:
import re, time, json, os
import torch
import pandas as pd
import numpy as np
import bs4

from langchain_community.document_loaders import PyMuPDFLoader, WikipediaLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_cohere import ChatCohere
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from rouge_score import rouge_scorer as rs
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
from sentence_transformers import CrossEncoder

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [3]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    COHERE_API_KEY = userdata.get('COHERE_API_KEY')
except ImportError:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    COHERE_API_KEY = os.environ.get('COHERE_API_KEY', '')

os.environ['HF_TOKEN'] = HF_TOKEN

## 2. Load Best Config from Notebook 1

In [4]:
# Load the best config exported from NB3 (Mistral optimization).
# Hardcoded from NB3 results — no file dependency needed.
best_cfg = {
    'chunk_size': 1500,
    'chunk_overlap': 300,
    'embedding_model': 'multi-qa-mpnet-base-dot-v1',
    'retriever_k': 8,
    'retriever_type': 'similarity',
    'splitter_type': 'recursive',
}
# Override from file if available
try:
    with open('best_mistral_config.json') as f:
        best_cfg = json.load(f)
        print('Loaded config from best_mistral_config.json')
except FileNotFoundError:
    print('Using hardcoded best config from NB3: chunk1500, k=8, similarity, mpnet, recursive')

CHUNK_SIZE = best_cfg['chunk_size']
CHUNK_OVERLAP = best_cfg['chunk_overlap']
EMBED_MODEL = best_cfg['embedding_model']
RETRIEVER_K = best_cfg['retriever_k']
RETRIEVER_TYPE = best_cfg['retriever_type']
print(f'Base config: chunk={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}, embed={EMBED_MODEL}, k={RETRIEVER_K}, type={RETRIEVER_TYPE}')

Using hardcoded best config from NB3: chunk1500, k=8, similarity, mpnet, recursive
Base config: chunk=1500, overlap=300, embed=multi-qa-mpnet-base-dot-v1, k=8, type=similarity


## 3. Load LLMs

In [5]:
# ---- Mistral ----
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16
)
llm_model = AutoModelForCausalLM.from_pretrained(
    'mistralai/Mistral-7B-Instruct-v0.3',
    torch_dtype=torch.float32, device_map='auto',
    quantization_config=quantization_config
)
llm_tokenizer = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-Instruct-v0.3')
mistral_pipe = pipeline(
    'text-generation', model=llm_model, tokenizer=llm_tokenizer,
    max_new_tokens=800, temperature=0.2, top_p=0.9,
    repetition_penalty=1.15, do_sample=True,
    pad_token_id=llm_tokenizer.eos_token_id
)
mistral_llm = HuggingFacePipeline(pipeline=mistral_pipe)
output_parser = StrOutputParser()

# ---- Cohere ----
cohere_llm = ChatCohere(cohere_api_key=COHERE_API_KEY)
print('LLMs loaded.')

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'temperature', 'top_p', 'repetition_penalty', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLMs loaded.


## 4. Load Documents (same corpus as NB1)

In [6]:
global_doc_number = 1

arxiv_numbers = (
    '2005.11401', '2104.07567', '2104.09864', '2105.03011', '2106.09685',
    '2203.02155', '2203.15556', '2211.09260', '2211.12561', '2212.09741',
    '2305.14314', '2305.18290', '2306.15595', '2309.08872', '2309.15217',
    '2310.06825', '2310.11511', '2311.08377', '2312.05708', '2401.06532',
    '2401.17268', '2402.01306', '2402.19473', '2406.04744', '2312.10997',
    '2410.12812', '2410.15944', '2411.16594', '2404.00657', '2601.07711',
    '2507.09477', '2506.10408'
)
all_arxiv_pages = []
for identifier in arxiv_numbers:
    pages = PyMuPDFLoader(f'https://arxiv.org/pdf/{identifier}.pdf').load()
    for page in pages:
        page.metadata['doc_num'] = global_doc_number
        page.metadata['doc_source'] = 'ArXiv'
        all_arxiv_pages.append(page)
    global_doc_number += 1

wiki_queries = [('Generative Artificial Intelligence',4),('Information Retrieval',2),
                ('Large Language Models',2),('Retrieval Augmented Generation',2)]
all_wiki_pages = []
for q, n in wiki_queries:
    try:
        wdocs = WikipediaLoader(query=q, load_max_docs=n).load()
        for d in wdocs:
            d.metadata['doc_num'] = global_doc_number
            d.metadata['doc_source'] = 'Wikipedia'
            global_doc_number += 1
        all_wiki_pages.extend(wdocs)
    except Exception as e:
        print(f'Wiki warning: {e}')

documents = WebBaseLoader(
    web_paths=('https://lilianweng.github.io/posts/2023-06-23-agent/',),
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_=('post-content','post-title','post-header')))
).load()
for d in documents:
    d.metadata['doc_num'] = global_doc_number
    d.metadata['doc_source'] = 'WWW'
    global_doc_number += 1

web_documents = WebBaseLoader(
    web_paths=(
        'https://lilianweng.github.io/posts/2020-10-29-odqa/',
        'https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/',
        'https://lilianweng.github.io/posts/2018-06-24-attention/',
        'https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/',
        'https://lilianweng.github.io/posts/2024-07-07-hallucination/',
        'https://huyenchip.com/2025/01/16/ai-engineering-pitfalls.html',
        'https://blog.langchain.com/the-rise-of-context-engineering/'
    ),
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_=('post-content','post-title','post-header')))
).load()
for d in web_documents:
    d.metadata['doc_num'] = global_doc_number
    d.metadata['doc_source'] = 'WWW'
    global_doc_number += 1

all_documents = all_arxiv_pages + all_wiki_pages + documents + web_documents
print(f'Total documents: {len(all_documents)}')

Total documents: 705


In [7]:
validation_questions_answers = {
0: {"question": "What pre-trained neural retriever model is used as the non-parametric memory component in RAG, providing latent documents conditioned on the input?",
  "gold_answer_research": "The pre-trained neural retriever model used as the non-parametric memory component in RAG is the Dense Passage Retriever (DPR). DPR provides latent documents conditioned on the input for the seq2seq model (BART) to generate the output. This retriever is initialized using retrieval supervision on Natural Questions and TriviaQA datasets, and it allows for efficient retrieval and incorporation of relevant information during the generation process.",
  "gold_answer_marketing": "The pre-trained neural retriever model used as the non-parametric memory component in RAG is the Dense Passage Retriever (DPR)."},
4: {"question": "How does the BM25 technique determine which passages are most relevant to a given query?",
  "gold_answer_research": "The BM25 technique determines which passages are most relevant to a given query by calculating a relevance score based on the frequency of query terms in each passage and the inverse document frequency. By using gold passages from the same mini-batch and one negative passage with a high BM25 score in combination, the retrieval results can be significantly improved. Additionally, exploring a setting where BM25 score and dense embedding retrieval score are linearly combined can serve as a new ranking function to further enhance the retrieval process.",
  "gold_answer_marketing": "The BM25 technique determines the relevance of passages by combining a score based on BM25 with a dense embedding retrieval score. This new ranking function helps improve retrieval results by using gold passages from the same mini-batch along with one negatively scored passage."},
5: {"question": "How does the RAG (retrieval-augmented generation) model leverage neural retrievers and document representation techniques to enhance the retrieval and generation process compared to earlier word-similarity-based methods?",
  "gold_answer_research": "The RAG model leverages a Dense Passage Retriever (DPR) that is pre-trained to rank correct passages in various question-answering settings. This allows for more accurate retrieval of relevant document chunks. The bi/dual-encoder nature of the DPR model enables effective document representation, improving the retrieval process compared to earlier word-similarity-based methods. By combining neural retrievers with document representation techniques, RAG enhances the overall retrieval and generation process by providing contextually relevant and factually accurate responses.",
  "gold_answer_marketing": "The RAG model combines neural retrievers, like the Dense Passage Retriever (DPR), with document representation techniques to improve retrieval and generation. This integration allows for contextually relevant and factually accurate responses compared to earlier word-similarity-based methods. The bi/dual-encoder nature of the DPR model enables better matching of documents and queries, enhancing the overall performance of the RAG framework."},
11: {"question": "How does the introduction of rotary position encoding (RoPE) in RoFormer influence the training convergence and evaluation metrics when compared to traditional position encoding methods in transformer-based models?",
  "gold_answer_research": "The introduction of rotary position encoding (RoPE) in RoFormer may improve training convergence and evaluation metrics compared to traditional position encoding methods in transformer-based models. RoPE offers valuable properties such as sequence length flexibility, decaying inter-token dependency with increasing relative distances, and the capability of equipping linear self-attention with relative position encoding. This may lead to better performance in terms of loss, accuracy, and overall model efficacy. Additionally, RoPE's multiplicative nature and incorporation of relative position information through rotation matrix product could contribute to more efficient training convergence and improved evaluation metrics in comparison to traditional additive position encoding methods.",
  "gold_answer_marketing": "The introduction of rotary position encoding (RoPE) in RoFormer improves training convergence and evaluation metrics compared to traditional position encoding methods in transformer-based models. This is evidenced by better performance results in terms of loss and accuracy across different training stages and tasks."},
13: {"question": "Which characteristics or innovations distinguish Longformer from other transformer-based models when processing lengthy texts?",
  "gold_answer_research": "Longformer stands out from other transformer-based models by its ability to efficiently process input sequences that are thousands of tokens long, specifically supporting input sequence lengths of up to 16K tokens. This capability allows Longformer to encode 99% of full texts in the QASPER dataset without the need for truncation. Additionally, Longformer-Encoder-Decoder (LED) offers valuable properties such as sequence length flexibility, decaying inter-token dependency with increasing relative distances, and the ability to equip linear self-attention with relative position encoding. These characteristics make Longformer a standout choice for handling lengthy text processing tasks efficiently.",
  "gold_answer_marketing": "Longformer differentiates itself by its support for processing input sequences up to 16K tokens long without truncation, a capability that enables it to encode 99% of full-text papers in the QASPER dataset. This is made possible by its Longformer-Encoder-Decoder (LED) architecture, which efficiently handles long inputs through its specialized design."},
15: {"question": "How does LoRA enable efficient adaptation of large Transformer-based language models while minimizing the number of parameters that need to be trained for downstream tasks?",
  "gold_answer_research": "LoRA enables efficient adaptation of large Transformer-based language models by injecting trainable rank decomposition matrices into each layer of the Transformer architecture. This greatly reduces the number of trainable parameters required for downstream tasks, allowing for high model quality with minimal memory requirements. By optimizing the pre-trained model weights and utilizing additional factorized projections, LoRA ensures effective task-switching and performance improvement without introducing additional latency during inference. Additionally, the flexibility to apply LoRA to specific weight matrices in the neural network allows for targeted parameter reduction while maintaining model scalability and task performance.",
  "gold_answer_marketing": "LoRA reduces the number of trainable parameters in large language models like GPT-3 and RoBERTa by injecting trainable rank decomposition matrices into each layer of the Transformer architecture. This allows for efficient adaptation to specific tasks while maintaining model quality and reducing GPU memory requirements. It outperforms traditional fine-tuning methods in terms of parameter efficiency and performance on a variety of tasks."},
16: {"question": "How does the Low-Rank Adaptation (LoRA) technique enable efficient adaptation of large pre-trained neural networks for various downstream tasks while minimizing both storage requirements and inference latency?",
  "gold_answer_research": "The Low-Rank Adaptation (LoRA) technique allows for efficient adaptation of large pre-trained neural networks by reducing the number of trainable parameters through the use of rank decomposition matrices, while keeping the pre-trained weights fixed. This not only minimizes storage requirements but also enables quick task-switching without introducing inference latency. By optimizing the rank decomposition matrices of the dense layers' change during adaptation, LoRA provides a computationally efficient way to fine-tune models for different downstream tasks while maintaining high model quality.",
  "gold_answer_marketing": "LoRA technique reduces the number of trainable parameters and memory requirements by using low-rank adaptation matrices while keeping pre-trained weights fixed. This allows for efficient adaptation of large pre-trained neural networks for different tasks without introducing inference latency. Additionally, LoRA enables quick task-switching and better interpretability of weight correlations during adaptation."},
20: {"question": "How are hyperparameters such as learning rate, batch size, and maximum sequence length chosen or varied when applying LoRA to different model architectures on benchmark tasks?",
  "gold_answer_research": "When applying LoRA to different model architectures on benchmark tasks, hyperparameters such as learning rate, batch size, and maximum sequence length are chosen or varied through a systematic hyperparameter search. For example, a hyperparameter search may involve exploring different values for LoRA dropout, LoRA r, and LoRA layers to optimize performance. Additionally, the learning rate is adjusted based on the chosen LoRA configuration and the specific model architecture being used. The batch size is typically kept consistent across tasks to ensure fair comparisons, while the maximum sequence length may vary depending on the requirements of each individual task. Overall, these hyperparameters are carefully tuned to achieve the best performance when applying LoRA to different model architectures on benchmark tasks.",
  "gold_answer_marketing": "Hyperparameters like learning rate, batch size, and sequence length are determined through a hyperparameter search process to optimize LoRA performance across different model architectures and benchmark tasks. This search involves exploring variables such as LoRA dropout, LoRA multiplier, and the specific layers to apply LoRA to, while keeping other factors like learning rate proportional to LoRA for consistency and effectiveness."},
22: {"question": "How does fine-tuning large language models with human feedback impact their ability to generate responses that better reflect user instructions and reduce undesirable outputs?",
  "gold_answer_research": "Fine-tuning large language models with human feedback can significantly improve their ability to generate responses that better reflect user instructions. This process helps align the language models with human intent and can reduce undesirable outputs, such as toxic or biased content. By incorporating human feedback during the fine-tuning process, the models can better generalize to new instructions and increase their usability. Additionally, fine-tuning with human feedback has been shown to improve performance in specialized content generation, ensuring accuracy and tone consistency in responses.",
  "gold_answer_marketing": "Fine-tuning large language models with human feedback helps align them with user intent, resulting in better responses that reflect instructions accurately. This process also helps in reducing the generation of undesirable outputs such as toxic or biased content."},
23: {"question": "Which two compilations of public NLP tasks were used to fine-tune GPT-3 for comparison with InstructGPT, as mentioned in the evaluation of model performance on the API prompt distribution?",
  "gold_answer_research": "The two compilations of public NLP tasks used to fine-tune GPT-3 for comparison with InstructGPT are the GLUE benchmark and WikiSQL. These tasks cover a wide range of natural language understanding and generation tasks, providing a diverse set of challenges for the models to evaluate their performance on the API prompt distribution. Additionally, the models were also evaluated on other tasks such as NL to SQL queries and SAMSum to assess their capabilities beyond the fine-tuning distribution.",
  "gold_answer_marketing": "The two compilations of public NLP tasks used to fine-tune GPT-3 for comparison with InstructGPT are SQuAD and DROP as mentioned in the evaluation of model performance on the API prompt distribution."},
27: {"question": "How does the use of counterfactual evaluation contribute to reducing bias in language models?",
  "gold_answer_research": "Counterfactual evaluation helps in reducing bias in language models by testing the model's ability to recognize and disregard known inaccuracies within documents, even when informed about potential misinformation. This approach evaluates the model's performance based on context relevance and noise robustness for retrieval quality, as well as answer faithfulness, relevance, negative rejection, information integration, and counterfactual robustness for generation quality. By analyzing relative probabilities and entropy, models can be assessed for bias, and improvements can be made to align their behavior with societal expectations and reduce harmful outputs. Various methods, such as adversarial setups and additional labeling, can be explored to decrease the generation of toxic or biased content.",
  "gold_answer_marketing": "Counterfactual evaluation helps reduce bias in language models by testing their ability to recognize and ignore known inaccuracies in documents, even when informed about potential misinformation. This approach is crucial for ensuring context relevance, noise robustness, and overall quality in retrieval and generation processes. It also provides insights into improving models' behavior to align more closely with user expectations."},
28: {"question": "What are some common ways in which GPT-3 is prompted to perform summarization, brainstorming, and classification tasks, based on the examples provided?",
  "gold_answer_research": "Some common ways in which GPT-3 is prompted to perform summarization tasks include instructions like \"Please help me summarize this paragraph into one sentence.\" For brainstorming tasks, users may ask GPT-3 to generate creative ideas based on a given context, such as \"Please give indie movie ideas like a guy traveling to South America to become a shaman.\" In classification tasks, GPT-3 may be tasked with determining sentiment from a tweet, for example, \"This is a tweet sentiment classifier, is the sentiment positive or negative?\" These prompts showcase the diverse range of tasks GPT-3 can perform, from summarization to brainstorming to classification.",
  "gold_answer_marketing": "GPT-3 is prompted for summarization tasks by asking it to condense lengthy texts into concise summaries. For brainstorming tasks, it generates creative ideas based on user instructions and context. In classification tasks, GPT-3 can be asked to classify sentiment or other attributes of text inputs."},
33: {"question": "How does adjusting the pretraining loss or KL reward coefficient affect the model’s performance on tasks such as DROP and SQuAD v2, and what trade-offs are observed in terms of validation reward?",
  "gold_answer_research": "Adjusting the pretraining loss coefficient has shown to be crucial in improving model performance on tasks like DROP and SQuAD v2. A value of 27.8 seems to work well across different model sizes, leading to significant improvements on these tasks without drastic drops in validation reward. On the other hand, increasing the KL reward coefficient can also improve performance, but it is essential to find the optimal balance, as values that are too low or too high can result in poor performance. Overall, there is a trade-off between task performance and validation reward when adjusting these coefficients, highlighting the need for careful tuning to achieve the best results.",
  "gold_answer_marketing": "Adjusting the pretraining loss coefficient can improve performance on tasks like DROP and SQuAD v2 with minimal impact on validation reward. Increasing the KL reward coefficient can also have an effect on performance, but finding the optimal balance is crucial as higher KL values may lead to performance regressions. Trade-offs between performance on specific tasks and overall validation reward need to be carefully considered when adjusting these coefficients."},
36: {"question": "How does the use of PPO-ptx and SFT compare in terms of their effectiveness at reducing undesired language generation when evaluated with RealToxicityPrompts?",
  "gold_answer_research": "Based on the human evaluation results, it appears that PPO with pretraining data mix is less sensitive to changes in learning rate. Therefore, the engineering group chose checkpoints with the highest likert scores as their final models. In terms of reducing undesired language generation when evaluated with RealToxicityPrompts, PPO-ptx and SFT were compared. PPO-ptx showed better performance in following explicit constraints, attempting correct instructions, and reducing 'hallucinations' compared to SFT. This indicates that PPO-ptx may be more effective at mitigating undesired language generation in this context.",
  "gold_answer_marketing": "Based on human evaluation results, PPO-ptx appears to be more effective at reducing undesired language generation compared to SFT when evaluated with RealToxicityPrompts. The choice of checkpoints with the highest likert scores determined PPO-ptx as the preferred model for this purpose."},
38: {"question": "How does the Sieve of Eratosthenes algorithm operate to efficiently generate a list of prime numbers up to a specified value, and what steps are taken to ensure only valid input is processed?",
  "gold_answer_research": "The Sieve of Eratosthenes algorithm efficiently generates a list of prime numbers up to a specified value by iteratively eliminating multiples of prime numbers. To ensure only valid input is processed, the algorithm first checks that the input is positive. It then declares a set of Boolean flags and initializes variables for storing primes and iterating through the sieve array. The algorithm marks non-prime numbers in the sieve array and appends prime numbers to the list, ensuring that only prime numbers are included in the final output.",
  "gold_answer_marketing": "The prime sieve function generates a list of prime numbers up to a specified value by eliminating multiples of primes. It ensures only valid input is processed by checking that the input is positive before creating a list of primes using Boolean flags."},
41: {"question": "How does the use of explicit natural language instructions alter the way a retrieval system determines which documents are relevant to a given query, compared to traditional retrieval methods that rely solely on queries or task identifiers?",
  "gold_answer_research": "The use of explicit natural language instructions in a retrieval system provides a more detailed and nuanced understanding of the user's search intent compared to traditional methods that rely solely on queries or task identifiers. By providing a natural language description of the search task, the retrieval system can better interpret and comprehend the user's information needs, leading to more accurate and relevant document retrieval. This approach allows for a more personalized and user-friendly retrieval process by explicitly modeling the user's search intent through instructions, ultimately enhancing the efficiency and effectiveness of the retrieval system.",
  "gold_answer_marketing": "Using explicit natural language instructions in retrieval systems helps define the relevance of documents to a query more precisely, considering diverse criteria beyond just the query itself. This method allows for a more general and task-aware approach to retrieval, enhancing the system's capability to match user expectations. Traditional methods that only rely on queries or task identifiers may not capture the full intent behind the user's search as effectively."},
44: {"question": "How have pretrained transformer models such as BERT influenced advancements in text ranking methods?",
  "gold_answer_research": "Pretrained transformer models like BERT have significantly advanced text ranking methods by incorporating additional knowledge through pre-training models. These models have enhanced language models by improving contextual representations, such as replacing original position encodings with RoPE during pre-training. Additionally, models like RoBERTa have optimized pre-training recipes and improved task performance without introducing many more trainable parameters, leading to better text ranking results. The use of transformers in text ranking has revolutionized NLP tasks and achieved state-of-the-art performance in various benchmarks.",
  "gold_answer_marketing": "Pretrained transformer models like BERT have revolutionized text ranking methods by enhancing language models with additional knowledge through pre-training techniques. They have improved performance in tasks such as reranking passages using rule-based or model-based approaches, resulting in more precise language model processing. The use of BERT and similar models has led to significant advancements in text ranking methods by optimizing dense retrieval model training with hard negatives and achieving high accuracy in identifying answer spans."},
54: {"question": "How does the Elo rating system contribute to the evaluation of chatbot models like those trained with QLORA, and what are its advantages compared to relying solely on performance scores or absolute scales?",
  "gold_answer_research": "The Elo rating system helps in evaluating chatbot models trained with QLORA by providing a measure of the expected win-rate relative to an opponent's win rate. This allows for a more nuanced comparison of model performance based on pairwise judgments from human annotators and GPT-4. The advantages of using Elo ratings include capturing the relative performance of chatbot models without the need for grounding an absolute scale, highlighting uncertainties in model-based evaluations, and enabling a more comprehensive ranking of chatbot performance based on tournament results.",
  "gold_answer_marketing": "The Elo rating system helps determine the ranking of chatbot models based on their expected win-rates against opponents. This system provides a relative measure of performance, allowing for comparisons between models regardless of the scale used for evaluation. This method offers a more nuanced and robust evaluation approach compared to simply looking at raw performance scores or traditional absolute scales."},
58: {"question": "How does the quality of instruction finetuning datasets affect the MMLU performance of language models compared to increasing the amount of training data or number of training epochs?",
  "gold_answer_research": "The quality of instruction finetuning datasets has a significant impact on MMLU performance compared to simply increasing the amount of training data or number of training epochs. Increasing dataset size and training for more epochs only marginally improves MMLU performance, while the difference between datasets can be up to 40 times larger in terms of MMLU. This indicates that dataset quality plays a crucial role in mean MMLU accuracy, highlighting the importance of selecting high-quality datasets for instruction finetuning.",
  "gold_answer_marketing": "The quality of the instruction finetuning datasets has a greater impact on MMLU performance than increasing the dataset size or training epochs. While more data and epochs marginally improve MMLU, the difference between datasets significantly affects accuracy. This highlights the critical role of dataset quality in achieving mean MMLU accuracy."},
72: {"question": "What is the name of the method of prompting the model that improves the ability of an LLM to respond to questions over structured documents?",
  "gold_answer_research": "The method of prompting the model that improves the ability of an LLM to respond to questions over structured documents is called PDFTriage. This method enhances the LLM's capability to provide accurate responses by treating documents as structured objects rather than plain text. By utilizing this method, the LLM is better equipped to understand and process information from structured documents, resulting in improved question-answering performance.",
  "gold_answer_marketing": "The method of prompting the model that improves the ability of an LLM to respond to questions over structured documents is called PDFTriage."},
73: {"question": "How does the PDFTriage technique utilize document structure to enhance the accuracy of answers provided by retrieval-augmented language models?",
  "gold_answer_research": "The PDFTriage technique utilizes the structured metadata of a PDF document to implement a more precise and accurate document question-answering approach. It generates a structured representation of the document by extracting information from section text, figure captions, headers, and tables. This structured metadata allows a retrieval-augmented language model to select the appropriate document frame needed to answer a query, enhancing the accuracy of the answers provided compared to existing approaches. By leveraging the document's structure, PDFTriage enables the model to retrieve context based on either structure or content, making it effective across different classes of questions where other retrieval-augmented models fail.",
  "gold_answer_marketing": "PDFTriage leverages a PDF's structured metadata to implement a more precise and accurate document question-answering approach by extracting information from section text, figure captions, headers, and tables. This structured metadata representation allows the model to select the appropriate document frame needed to answer queries, improving answer quality and accuracy."},
80: {"question": "How does the sliding window attention mechanism contribute to both efficiency and information propagation in the Mistral 7B transformer architecture?",
  "gold_answer_research": "The sliding window attention mechanism in Mistral 7B allows each token to attend to at most W tokens from the previous layer, even though tokens outside the sliding window still influence next word prediction. This mechanism helps improve efficiency by reducing the latency and smaller throughput due to reduced cache availability. Furthermore, the information can move forward by up to k x W tokens after k attention layers, contributing to enhanced information propagation within the transformer architecture.",
  "gold_answer_marketing": "The sliding window attention in Mistral 7B allows tokens to attend to a limited number of tokens from previous layers, improving efficiency. This mechanism also enables information to propagate forward by a certain number of tokens at each attention layer, enhancing information flow within the architecture. Ultimately, this helps improve performance and reduce computational costs in Mistral 7B."},
83: {"question": "How does Self-reflective Retrieval-Augmented Generation improve the reliability and verifiability of generated text compared to standard RAG approaches?",
  "gold_answer_research": "Self-Reflective Retrieval-Augmented Generation (SELF-RAG) enhances reliability and verifiability by training an arbitrary LM to generate text with reflection tokens for self-assessment. This allows the model to reflect on whether the output is supported by the retrieved passages, leading to easier fact verification. Unlike standard RAG approaches that may indiscriminately retrieve and incorporate irrelevant or unhelpful passages, SELF-RAG's self-reflective nature ensures that generated text is more aligned with the provided knowledge and avoids factual inaccuracies commonly seen in conventional approaches. Additionally, SELF-RAG provides citations for each segment, further improving the reliability and traceability of the generated text.",
  "gold_answer_marketing": "Self-reflective Retrieval-Augmented Generation (SELF-RAG) improves reliability and verifiability of generated text by training a single LM to adaptively retrieve and reflect on passages on-demand, ensuring factual accuracy and reducing inaccuracies. This approach enhances the generation quality by providing citations for each segment and self-assessment of output based on supported passages, aiding in easier fact verification."},
90: {"question": "Which named technique involves mapping words or phrases from the vocabulary to vectors of real numbers in Natural Language Processing (NLP)?",
  "gold_answer_research": "One named technique that involves mapping words or phrases from the vocabulary to vectors of real numbers in NLP is Word Embedding. Word embedding is a set of language modeling and feature learning techniques where words or phrases are represented as continuous vector spaces with a much lower dimension. This method helps capture similarities between words and their context in NLP tasks.",
  "gold_answer_marketing": "Word embedding involves mapping words or phrases from the vocabulary to vectors of real numbers in Natural Language Processing (NLP)."},
91: {"question": "How has the architectural paradigm of RAG evolved from its earliest implementations to current state-of-the-art frameworks?",
   "gold_answer_research":  "The RAG architecture initially emerged as a \"Naive RAG\" paradigm consisting of a straightforward, linear sequence of indexing documents, retrieving relevant chunks, and generating text. Over time, this evolved into \"Advanced\" and \"Modular RAG\" systems that incorporated specialized components for pre-retrieval query rewriting, post-retrieval reranking, and flexible routing among multiple data sources. Most recently, the field has shifted toward \"Agentic RAG,\" which replaces fixed pipelines with autonomous systems where an LLM orchestrates the entire process, deciding when to fetch tools, iteratively refining queries, and reflecting on its own generations.",
   "gold_answer_marketing": "RAG systems began as simple pipelines that retrieved relevant documents and used them to generate answers. Today’s systems are far more sophisticated, using multiple data sources and AI “agents” that can decide what information to retrieve and how to refine answers to produce more accurate and useful results."},
93: {"question": "How does the use of re-ranking methods such as LambdaMART influence the retrieval of relevant context for LLM-based planning, particularly in comparison to using off-the-shelf retrievers or query augmentation techniques?",
  "gold_answer_research": "The use of re-ranking methods like LambdaMART can significantly impact the retrieval of relevant context for LLM-based planning by improving the ranking of candidate documents according to their relevance to the user's query. This approach enhances contextual understanding of implicit and ambiguous queries, which is crucial for plan generation. Compared to off-the-shelf retrievers or query augmentation techniques, re-ranking methods like LambdaMART have been shown to outperform in terms of performance and efficiency, especially when dealing with complex queries that demand context-seeking capabilities. It provides a more targeted and accurate approach to document reranking, leading to better outcomes in LLM-based planning tasks.",
  "gold_answer_marketing": "The use of re-ranking methods like LambdaMART enhances the retrieval of relevant context for LLM-based planning by improving the ranking of candidate documents based on their relevance to the user's query. Compared to off-the-shelf retrievers or query augmentation techniques, re-ranking with LambdaMART has been shown to outperform in terms of performance and efficiency."},
94: {"question": "How does context tuning impact the effectiveness of RAG-based planning systems compared to traditional semantic search approaches, particularly in terms of plan accuracy and reduction of hallucinations?",
  "gold_answer_research": "Context tuning in RAG-based planning systems significantly enhances contextual understanding and improves tool retrieval compared to traditional semantic search approaches. This leads to higher plan accuracy as context tuning helps in providing relevant and accurate responses to complex requests. Additionally, context tuning reduces hallucinations by ensuring that plans are generated using real-world context and information, thereby improving the overall effectiveness of RAG-based planning systems.",
  "gold_answer_marketing": "Context tuning in RAG-based planning systems improves plan accuracy and reduces hallucinations compared to traditional semantic search approaches. It enhances contextual understanding and context seeking abilities, leading to more relevant tool retrieval and improved plan generation. This ultimately results in better performance and accuracy in knowledge-intensive tasks."},
95: {"question": "How can the use of context compression techniques benefit retrieval-augmented generation models, particularly in terms of balancing performance improvements with resource limitations?",
  "gold_answer_research": "Context compression techniques can benefit retrieval-augmented generation models by allowing for the reduction of unnecessary information, thus improving the efficiency of the generation process. By compressing context, models can focus on the most relevant and useful information, balancing performance improvements with resource limitations. This helps in mitigating challenges such as slow generation process due to lengthy context and trade-offs in accuracy or costs. Additionally, context compression techniques can optimize the model's ability to filter relevant content and generate output, ultimately improving the overall performance of the retrieval-augmented generation system.",
  "gold_answer_marketing": "Context compression techniques can benefit retrieval-augmented generation models by improving performance without requiring excessive computational resources. These techniques help balance the trade-off between accuracy and costs, allowing models to handle longer sequences efficiently. This results in faster generation processes and optimized resource usage for better overall performance."},
102: {"question": "What are the primary methodologies and potential pitfalls for evaluating RAG systems in scenarios where ground-truth reference answers are unavailable?",
   "gold_answer_research":  "In the absence of human-annotated ground truths, developers often employ reference-free evaluation frameworks like Ragas, which utilize LLMs to automatically measure metrics such as answer faithfulness, context relevance, and answer relevance. Alternatively, system evaluators can adopt an \"LLM-as-a-judge\" approach to assign continuous scores, rank candidate outputs, or verify the presence of hallucinations via automated comparisons. However, a major pitfall in relying exclusively on automated AI judges is that these models are not perfectly deterministic and can produce misleading evaluations, underscoring the critical need to continually validate automated scoring against systematic human reviews.",
   "gold_answer_marketing": "When there are no “correct” answers to compare against, teams often use AI tools to evaluate whether responses are relevant, accurate, and grounded in the retrieved information. However, because AI-based evaluation can sometimes be inconsistent or misleading, it’s important to periodically validate those results with human review."},
103: {"question": "How does query expansion help search engines improve the relevance of documents returned to users?",
  "gold_answer_research": "Query expansion helps search engines improve the relevance of documents returned to users by elaborating original queries into more detailed versions while preserving the search intent. This process enhances the search engine's understanding of the user's needs, leading to more accurate and relevant document retrieval. By adding more context and depth to the query, query expansion helps the search engine retrieve documents that align more closely with what the user is looking for, ultimately improving the overall search results and user experience.",
  "gold_answer_marketing": "Query expansion helps search engines by elaborating original queries into more detailed versions, preserving the search intent. This process enhances the search engine's understanding of user needs, leading to more accurate and relevant document retrieval."},
108: {"question": "What is the recommended strategy for training large autoregressive language models with limited compute resources, as contributed by the Chinchilla team?",
  "gold_answer_research": "The Chinchilla team recommends that the number of training tokens should be doubled for every model size doubling to achieve better results on downstream tasks. They also suggest using larger, higher-quality training datasets to improve performance. Additionally, they mention the importance of balancing model size and efficiency to address computational costs and inference latency limitations. It is advised to focus on Transformer language models and consider sharing model parameters for quick task-switching when deploying as a service.",
  "gold_answer_marketing": "The Chinchilla team recommends doubling the number of training tokens for every model size doubling and using larger, higher-quality training datasets to achieve better results on downstream tasks."},
111: {"question": "What optimizer and default learning rate are recommended for KTO in practice?",
  "gold_answer_research": "For KTO in practice, it is recommended to start with the optimizer AdamW and a default learning rate of 5e-6. This learning rate should be adjusted as needed based on the performance of the model. It is important to remember that the performance of the aligned model is highly sensitive to the learning rate, so experimentation may be necessary to find the optimal rate for your specific application.",
  "gold_answer_marketing": "For KTO in practice, it is recommended to use the AdamW optimizer with a starting learning rate of 5e-6. This learning rate can be adjusted as needed for optimal performance."},
112: {"question": "How does the technique of reinforcement learning from human feedback (RLHF) leverage collected human preferences to improve the behavior of language models?",
  "gold_answer_research": "Reinforcement learning from human feedback (RLHF) leverages collected human preferences by first fitting a reward model to reflect these preferences. The RLHF technique then optimizes a language model policy to produce responses that align with high-reward preferences without deviating too far from the original model. This process allows language models to better understand user intentions, align with human preferences, and reduce the overall cost of communication. Additionally, RLHF helps improve the model's conversational abilities, coding proficiency, factuality, and task alignment by learning from human or automated preference judgments.",
  "gold_answer_marketing": "RLHF leverages collected human preferences to improve language models by fitting a reward model to human preferences and using reinforcement learning to optimize the language model policy. This helps the language model produce responses aligned with human intentions while minimizing drift from the original model."},
118: {"question": "How is external knowledge or information from retrieved texts, captions, or demonstrations typically incorporated into vision-and-language or multi-modal generation models to enhance their outputs?",
  "gold_answer_research": "External knowledge or information from retrieved texts, captions, or demonstrations is typically incorporated into vision-and-language or multi-modal generation models by allowing the model to refer to relevant examples from an external memory when generating images or text. This augmentation helps reduce the need for storing all knowledge inside the model, resulting in a more efficient use of parameters and training data. Additionally, models can use in-context examples for controlled generation, providing valuable guidance for generating accurate and detailed outputs. By accessing external memory, models can accommodate the growth and update of knowledge over time, leading to improved outputs in tasks like image captioning and text-to-image generation.",
  "gold_answer_marketing": "External knowledge is typically incorporated into vision-and-language or multi-modal generation models by augmenting them with the ability to refer to relevant examples from an external memory. This allows the models to access additional knowledge beyond what is stored inside the model itself, improving the quality of outputs by incorporating multi-modal information and enhancing dialog generation with external knowledge."},
119: {"question": "How does Retrieval Augmented Generation (RAG) maintain its effectiveness in handling both recent and infrequently discussed knowledge, especially in comparison to long-context models?",
  "gold_answer_research": "Retrieval Augmented Generation (RAG) maintains its effectiveness in handling both recent and infrequently discussed knowledge by leveraging a flexible data repository that acts as a non-parametric memory, allowing for easy updates and accommodating extensive long-tail knowledge. This approach mitigates the lack of long-tail knowledge and can encode confidential data. Additionally, RAG's retrieval capabilities reduce generation costs, support long contexts, and provide opportunities for addressing complex problems and integrative or summary questions that require reading a large amount of material to answer.",
  "gold_answer_marketing": "Retrieval Augmented Generation (RAG) maintains effectiveness by leveraging a flexible data repository that can easily update and store extensive long-tail knowledge, including confidential data. This non-parametric memory allows RAG to address complex problems by retrieving relevant passages from a large corpus, supporting long contexts, and reducing generation costs."},
123: {"question": "How does the automatic evaluation process using ChatGPT and Llama 3 distinguish between partially correct, completely correct, and incorrect responses, and how are these classifications reflected in their accuracy and F1 scores?",
  "gold_answer_research": "The automatic evaluation process using ChatGPT and Llama 3 distinguishes between partially correct, completely correct, and incorrect responses by first checking if the answer matches the ground truth exactly. If it does, it is considered accurate. If not, the responses are further evaluated by the two LLM evaluators to determine if they are accurate, incorrect, or missing. These classifications are reflected in their accuracy and F1 scores by showing different performance metrics for accurate, incorrect, and missing answers in both models, with Llama 3 generally outperforming ChatGPT in terms of precision, recall, and F1 score for accurate responses.",
  "gold_answer_marketing": "The automatic evaluation process using ChatGPT and Llama 3 distinguishes between accurate, incorrect, and missing responses by comparing them to the ground truth. This classification is reflected in their accuracy and F1 scores, with ChatGPT showing an average F1 score of 94.7% and Llama 3 showing an average F1 score of 98.9% compared to human evaluation."},
124: {"question": "What are some of the main challenges that Retrieval-Augmented Generation aims to address when applied to large language models, and how does integrating external knowledge bases help overcome these obstacles?",
  "gold_answer_research": "Retrieval-Augmented Generation (RAG) aims to address challenges such as hallucination, outdated knowledge, and non-transparent reasoning processes in large language models. By incorporating knowledge from external databases, RAG enhances accuracy and credibility in generation tasks, particularly for knowledge-intensive tasks. The integration of external knowledge bases helps provide real-time, relevant information to the generative model, improving the quality of the outputs and mitigating issues related to outdated or erroneous information in the model itself.",
  "gold_answer_marketing": "Retrieval-Augmented Generation aims to address challenges like hallucination, outdated knowledge, and non-transparency in large language models. By incorporating external knowledge bases, RAG enhances accuracy, credibility, and the overall quality of generated outputs for knowledge-intensive tasks."},
126: {"question": "How do different Retrieval Augmented Generation (RAG) methods vary in the stage at which retrieval is integrated into the pipeline, and what are some examples of techniques that incorporate retrieval during model pre-training versus tuning or inference?",
  "gold_answer_research": "Different RAG methods vary in the stage at which retrieval is integrated into the pipeline. Some techniques incorporate retrieval during model pre-training, where the retrieval component is jointly trained with the generative model from scratch. An example of this approach is denoising objectives similar to BART. On the other hand, some methods incorporate retrieval during tuning or inference stages, where a pre-trained generative model is augmented with retrieval components. An example of this is query-based RAG, which allows for modular flexibility and quick integration of pre-trained components during deployment.",
  "gold_answer_marketing": "Different Retrieval Augmented Generation (RAG) methods vary in when retrieval is integrated into the pipeline, with some incorporating retrieval during model pre-training (e.g., for latent representation-based RAG) and others during tuning or inference (e.g., query-based RAG). Examples of techniques include iterative retrieval during generation for richer context, recursive retrieval for breaking down complex problems, and input enhancement by feeding retrieved content during the input phase."},
128: {"question": "What are the computational and practical trade-offs associated with deploying highly autonomous Agentic RAG systems compared to simpler, fixed-pipeline Enhanced RAG systems?",
  "gold_answer_research":  "While Agentic RAG systems offer high flexibility by allowing the language model to autonomously orchestrate multiple retrieval and reasoning steps, this autonomy significantly increases monetary costs, token usage, and end-to-end latency. Empirical comparisons show that Agentic RAG can be several times more expensive and slower than Enhanced RAG pipelines, often without yielding universally superior retrieval performance in noisier domains. Consequently, experts warn against prematurely adopting complex agentic frameworks when simpler, highly optimized pipelines utilizing explicit re-rankers or straightforward API calls would adequately suffice for the specific product requirements.",
   "gold_answer_marketing": "Agentic RAG systems can be more flexible because they allow AI to decide how to search for and combine information, but that flexibility often comes with higher costs and slower response times. In many cases, simpler RAG pipelines can deliver similar results more efficiently, so organizations should weigh the added complexity against the actual benefits."},
129: {"question": "Which automated tool employs LLMs to adjudicate quality scores for the evaluation of RAG models alongside ARES and TruLens?",
  "gold_answer_research": "The automated tool RAGAS employs LLMs to adjudicate the quality scores for the evaluation of RAG models, in addition to ARES and TruLens. These tools, along with benchmarks, create a robust framework for systematic evaluation of RAG models, as shown in Table IV. This approach can enhance the evaluation process by utilizing LLMs as a critical sample selector based on specific criteria for human annotators to evaluate.",
  "gold_answer_marketing": "RAGAS is the automated tool that, alongside ARES and TruLens, uses LLMs to adjudicate quality scores for evaluating RAG models."},
130: {"question": "What are some important considerations and recent advancements involved in making Retrieval Augmented Generation (RAG) systems suitable for use in real-world, production environments?",
  "gold_answer_research": "Some important considerations for making RAG systems suitable for real-world production environments include ensuring that the systems are developed and managed independently without conflicts or dependency issues, maintaining cleaner and more manageable development workflows, updating knowledge, handling long-tail data, mitigating data leakage, managing high training and inference costs, and integrating RAG within Large Language Models (LLMs) to enhance accuracy. Recent advancements in RAG systems focus on integrating information retrieval processes to enhance generation processes, pulling in information from external data sources such as PDFs, databases, or websites to ground generated content in accurate and current data. Additionally, recent research is focusing on methods to enhance the factual grounding of RAG outputs and analyzing costs and computational time required by different systems under various scenarios to help researchers and practitioners select the most appropriate system for their needs.",
  "gold_answer_marketing": "Considerations for making RAG systems suitable for production include managing high training and inference costs, updating knowledge, handling long-tail data, and mitigating data leakage. Recent advancements include the integration of RAG within Large Language Models, resulting in higher accuracy, better decision making, and enhanced capabilities in knowledge-intensive tasks. RAG systems retrieve relevant information from external sources, such as PDFs and databases, to ground generated content in accurate and current data for real-time applications."},
133: {"question": "Which framework is described as An Introspection Platform for RAG Evaluation?",
  "gold_answer_research": "The framework described as an introspection platform for RAG evaluation is Self-Reflective Retrieval-Augmented Generation (SELF-RAG). It enhances the quality and factuality of a Language and Logic Model (LLM) through retrieval and self-reflection, while maintaining the original creativity and versatility of the LLM. The framework leverages LRMs within RAG to improve information synthesis and knowledge retrieval effectively, with the model's inherent reasoning capabilities driving the process.",
  "gold_answer_marketing": "The framework described as an Introspection Platform for RAG Evaluation is SELF-RAG."},
134: {"question": "What benefits does Retrieval Augmented Generation (RAG) offer over traditional language models with respect to providing reliable and up-to-date answers, and how does its approach contribute to these improvements?",
  "gold_answer_research": "Retrieval Augmented Generation (RAG) models offer benefits over traditional language models by enhancing the factual grounding of their outputs through the explicit retrieval of relevant passages from a large corpus. This approach ensures that the answers provided are based on up-to-date and accurate knowledge. By integrating information retrieval with natural language generation, RAG models can provide reliable answers by incorporating real-time information and reducing the risk of producing incorrect or outdated responses. This dual-component framework contributes to improving the accuracy and credibility of the generated answers, particularly in knowledge-intensive domains, making it a promising solution for QA tasks.",
  "gold_answer_marketing": "Retrieval Augmented Generation (RAG) improves on traditional language models by explicitly retrieving up-to-date information from large databases, enhancing the accuracy and reliability of answers. This approach ensures that responses are contextually relevant and factually accurate, making it a valuable tool for knowledge-intensive tasks like question answering."},
135: {"question": "Why are documents in PDF format considered particularly valuable when implementing Retrieval Augmented Generation (RAG) systems?",
  "gold_answer_research": "Documents in PDF format are considered particularly valuable when implementing Retrieval Augmented Generation (RAG) systems because they contain dense and detailed information essential for training RAG models. PDFs are widely used for distributing high-value content like research papers, legal documents, technical manuals, and financial reports. Moreover, PDFs come in various forms, allowing access to a wide range of data types, from scientific data and technical diagrams to textual information. The complex formatting of PDFs poses challenges for accurate text extraction, making it essential to use reliable tools and methods for converting the PDF content into usable text for effective retrieval and generation.",
  "gold_answer_marketing": "PDF documents are essential for RAG applications because they contain dense, detailed information necessary for training RAG models. They are widely used for distributing high-value content like research papers, legal documents, technical manuals, and financial reports, making them ideal for knowledge-intensive tasks. PDFs allow access to a wide range of data types, from scientific data to technical diagrams, enhancing the accuracy and effectiveness of retrieval and generation processes."},
136: {"question": "What are some ways in which a Retrieval Augmented Generation (RAG) system ensures that the responses it generates are both contextually relevant and based on up-to-date information, especially when integrating knowledge from PDF documents?",
  "gold_answer_research": "One way a Retrieval Augmented Generation (RAG) system ensures contextually relevant and up-to-date responses is by integrating query-based RAG, which combines the user's query with insights from retrieved information to create a response. Additionally, RAG systems use dense retrieval methods to explicitly retrieve relevant passages from a large corpus, ensuring that the generated content is grounded in accurate and current data. By combining information retrieval techniques with generative models, RAG systems can provide responses that are not only contextually appropriate but also up-to-date, particularly when pulling information from domain-specific PDF documents.",
  "gold_answer_marketing": "A Retrieval Augmented Generation (RAG) system ensures contextually relevant and up-to-date responses by pulling in information from external data sources like PDFs, databases, or websites. This grounding in accurate and current data makes RAG ideal for knowledge intensive tasks, ensuring that the generated content is both relevant and up-to-date."},
139: {"question": "Because initial user queries are often ambiguous or overly brief, what techniques have been developed to transform and optimize these queries before they are sent to the retriever?",
  "gold_answer_research":  "To bridge the semantic gap between short user queries and extensive document text, RAG systems employ query transformation techniques that use language models to rewrite, expand, or clarify the original input. Methods like HyDE prompt the model to generate a hypothetical answer first, utilizing this pseudo-document's embedding to search for real, semantically similar documents in the database. Furthermore, advanced systems like DeepRetrieval utilize reinforcement learning to train models to iteratively rewrite queries based on the actual performance metrics of the downstream retrieval system, ensuring the search strings are highly optimized for fetching evidence.",
   "gold_answer_marketing": " Because users often ask short or unclear questions, modern RAG systems use AI to rewrite or expand the query so it better matches the information stored in the knowledge base. These techniques help the system retrieve more relevant content and produce more accurate answers."},
140: {"question": "How is the RAG architecture specifically adapted to support software engineering tasks such as code generation and automatic program repair?",
    "gold_answer_research": "In software engineering contexts, RAG systems augment code generation by retrieving relevant API documentations, similar code examples, and abstract syntax trees (ASTs) to construct highly contextualized prompts for the language model. For tasks like automatic program repair, hybrid retrieval methods combine sparse and dense searches to locate similar historical buggy codes and their corresponding patches, feeding this data into sequence-to-sequence models to synthesize functional fixes. Because natural language queries often differ structurally from programming languages, specialized adaptations like execution feedback loops, AST-based retrieval, and instruction-tuned code encoders are crucial for bridging the semantic gap in code-related RAG applications.",
  "gold_answer_marketing": "In software engineering, RAG systems improve code generation by retrieving relevant documentation, code examples, and other technical context to guide the AI’s output. They can also help fix bugs by finding similar past coding issues and solutions, giving the AI examples it can use to generate a working repair."},
143: {"question": "How do retrieval-augmented generation models handle information from multiple languages or different types of data modalities when providing responses?",
  "gold_answer_research": "Retrieval-augmented generation models handle information from multiple languages or different data modalities by utilizing a retriever to gather relevant documents from external memory, which are then used by a generator to create predictions based on the retrieved documents. While these methods were initially designed for text, extending them to a multimodal setting presents challenges that require designing a suitable retriever and generator architecture. Recent research has shown progress in applying retrieval-augmented generation to task-oriented dialogues and improving accuracy with semantic search and hybrid query-based retrievers. Further advancements in this area aim to enhance the model's ability to handle diverse linguistic and modal information for more robust and effective responses.",
  "gold_answer_marketing": "Retrieval-augmented generation models use a retriever to gather relevant information across languages or data types before generating responses. By combining retrieved documents with the original query, these models can provide more comprehensive and accurate outputs. This approach helps ensure that responses are informed by a diverse range of information sources."},
145: {"question": "Which technique proposed by Zheng et al. (2023) involves invoking the judge LLM twice and swapping the order of two candidates to mitigate positional bias in LLM-based evaluation systems?",
  "gold_answer_research": "The technique proposed by Zheng et al. (2023) to mitigate positional bias in LLM-based evaluation systems involves a swapping operation. This operation includes invoking the judge LLM twice and swapping the order of the two candidates in each instance. If the results from the two invocations are inconsistent, it is labeled a \"tie,\" indicating the LLM's inability to confidently distinguish the quality of the candidates. This swapping operation technique is widely adopted in pairwise feedback synthesis to improve the accuracy of reward signals.",
  "gold_answer_marketing": "The technique proposed by Zheng et al. (2023) is the swapping operation, which involves invoking the judge LLM twice and swapping the order of the two candidates to mitigate positional bias in LLM-based evaluation systems."},
146: {"question": "How can large language models be utilized to improve the process of selecting or filtering retrieved content in retrieval-augmented generation, and what specific benefits does this provide compared to traditional retrieval methods?",
  "gold_answer_research": "Large language models can be utilized in retrieval-augmented generation by using their advanced natural language processing capabilities to understand and interpret the retrieved content. This allows for more precise selection or filtering of relevant information, improving the quality and accuracy of the generated output. Compared to traditional retrieval methods, this approach provides the benefit of leveraging contextual understanding and semantic relationships, resulting in more informed predictions and better performance in domain-specific or knowledge-intensive tasks.",
  "gold_answer_marketing": "Large language models can be used in retrieval-augmented generation to enhance the selection or filtering of retrieved content by providing more relevant and current information. This approach offers benefits like improved accuracy, efficiency, and effectiveness compared to traditional retrieval methods."},
150: {"question": "How can the use of large language models as evaluators influence the assessment of natural language generation tasks, and what are some of the approaches used to ensure their judgments align with those of human annotators?",
  "gold_answer_research": "The use of large language models (LLMs) as evaluators can significantly impact the assessment of natural language generation tasks by providing automated scoring and ranking capabilities. To ensure that their judgments align with those of human annotators, approaches such as conducting human studies to justify the use of LLMs for evaluation, measuring performance on a range of language understanding tasks, and testing generative language capabilities through automated and human evaluations can be employed. Additionally, utilizing queries curated by humans and comparing inter-human annotator agreement with LLM judgments can help validate the effectiveness of LLMs in evaluating natural language generation tasks.",
  "gold_answer_marketing": "Large language models (LLMs) can provide automated evaluations for natural language generation tasks, but human validation is still necessary for accuracy. Approaches such as comparing LLM judgments with human agreement and using curated queries help align LLM assessments with human annotators."},
151: {"question": "Which benchmark is described as a hierarchical and comprehensive safety benchmark for large language models?",
  "gold_answer_research": "The hierarchical and comprehensive safety benchmark for large language models described in the given context is called WriteBench. It is designed to assess the writing capabilities of LLMs across multiple domains and tasks, ensuring a fair comparison between Weaver and other generalist LLMs. WriteBench will be publicly available for evaluation purposes on GitHub at https://github.com/aiwaves-cn/WriteBench.",
  "gold_answer_marketing": "The Massively Multitask Language Understanding (MMLU) benchmark is described as a hierarchical and comprehensive safety benchmark for large language models."},
154: {"question": "How does the Tree of Thoughts approach leverage large language models to enhance problem-solving capabilities compared to more straightforward reasoning techniques?",
  "gold_answer_research": "The Tree of Thoughts approach extends traditional reasoning methods by exploring multiple possibilities at each step, creating a tree structure that allows for branching logical pathways. This enables the model to consider multiple reasoning trajectories simultaneously, avoiding traps caused by early mistaken assumptions. By promoting a step-by-step thinking process, the Tree of Thoughts technique enhances the problem-solving capabilities of large language models, enabling them to deliberate and solve complex tasks more effectively than traditional, linear methods.",
  "gold_answer_marketing": "The Tree of Thoughts approach breaks down problems into multiple steps and generates multiple thoughts per step, allowing for exploration of different reasoning possibilities. By creating a tree structure and utilizing search algorithms like BFS or DFS, it enables more in-depth evaluation of problem-solving paths, enhancing the capabilities of large language models."},
155: {"question": "How does the self-discover approach utilize large language models to autonomously generate reasoning strategies?",
  "gold_answer_research": "The self-discover approach leverages large language models (LLMs) like SELF-RAG to autonomously generate reasoning strategies by incorporating self-reflection mechanisms. Through self-reflection, the LLM can assess its knowledge limitations and determine when to retrieve relevant information, enabling autonomous decision-making in generating responses. This approach eliminates the need for external models during inference, streamlining the decision-making process and improving the model's output quality, including factuality evaluation. Additionally, by integrating reasoning and tool use, the LLM can iteratively refine its responses based on newly retrieved information, enhancing its overall performance in complex natural language tasks.",
  "gold_answer_marketing": "The self-discover approach empowers large language models to introspectively evaluate their own knowledge gaps and retrieve information only when necessary. Unlike traditional approaches, this method does not rely on external models for inference, allowing the model to actively manage retrieval autonomously. This process enhances reasoning capabilities and enables the model to refine its responses iteratively based on newly retrieved information."},
157: {"question": "How do existing evaluation frameworks designed for LLM-as-a-judge differ in the types of human and automated metrics they utilize to assess general performance, bias, and robustness across conversation, alignment, and multimodal tasks?",
  "gold_answer_research": "Existing evaluation frameworks for LLM-as-a-judge differ in the types of metrics they use to assess general performance, bias, and robustness. Some frameworks focus on metrics like Cohen's kappa, Discernment Score, and normalized accuracy to evaluate alignment between LLM predictions and manual judgments in conversation and alignment tasks. Others, like ALLURE proposed by Hasanbeig et al., aim to enhance evaluator robustness by iteratively incorporating demonstrations of significant deviations. Additionally, frameworks like MLLM-as-a-judge by Dongping Chen et al. assess multimodal tasks using vision-language benchmarks, showcasing a diversity of metrics and approaches in LLM evaluation.",
  "gold_answer_marketing": "Existing evaluation frameworks for LLM-as-a-judge vary in the types of metrics used to assess general performance, bias, and robustness across conversation, alignment, and multimodal tasks. They may include automated metrics like Cohen's kappa, Discernment Score, and normalized accuracy, as well as human judgment-based metrics to evaluate the overall competence of LLMs in various tasks. Additionally, some approaches incorporate demonstrations of significant deviations to enhance evaluator robustness, while others leverage insights from many-shot in-context learning for more thorough and fair assessments."},
160: {"question": "When conducting demographic and technical assessments of teams or research subjects, what types of data categories are typically collected and analyzed to ensure a comprehensive understanding of the group's composition and the methods used?",
	"gold_answer_research": "When conducting demographic and technical assessments of teams or research subjects, it is important to collect and analyze data categories such as age, gender, education level, professional background, and expertise in specific areas. By gathering information on these categories, you can ensure a comprehensive understanding of the group's composition and the methods used in your assessments. Additionally, it may be helpful to consider factors like cultural background, language proficiency, and geographical location to capture a more nuanced picture of the group being assessed. This detailed approach to data collection and analysis can provide valuable insights for making informed decisions and recommendations based on the gathered information.",
	"gold_answer_marketing": "Demographic data such as age, gender, education level, and technical data related to skills and experience are typically collected and analyzed for comprehensive understanding."
    },
161: {"question": "How does the evaluation process determine the level of alignment in the models being tested?",
    "gold_answer_research": "The evaluation process determines the level of alignment in the models by comparing the system-level judgments made by the language model (GPT-4) with those made by human annotators, using metrics such as Kendall Tau and Spearman rank correlation. At the example level, the agreement between the model and human annotators is measured using Fleiss κ. These metrics provide insights into the alignment of the language model with human intentions and indicate the reliability of model-based evaluation as an alternative to human evaluation.",
    "gold_answer_marketing": "The evaluation process determines the level of alignment in the models being tested by comparing system-level judgments by the language model and human annotators, as well as examining agreement at the example level."
  },
162: {"question": "What approaches can be used to improve the performance of an AI model on various datasets, and how do they compare in terms of effectiveness?",
    "gold_answer_research": "To improve the performance of an AI model on various datasets, you can consider combining your method with ways to filter the pretraining data or training the initial pretrained models using human data. Additionally, you could explore methods that improve models' truthfulness, such as WebGPT. Comparing the effectiveness of these approaches, it may be beneficial to experiment with different adaptation approaches in the low-data regime and evaluate their performance on tasks like MNLI-n. Fine-tuning models using different training datasets and varying the number of fine-tuning steps can also impact performance on public NLP datasets.",
    "gold_answer_marketing": "Approaches such as fine-tuning with different datasets, filtering pretraining data, and combining methods to improve model truthfulness can be used to enhance the performance of an AI model on various datasets. However, the effectiveness of these approaches may vary depending on the specific task and dataset."
  },
165: {"question": "What are some common strategies for addressing the ethical and social risks associated with the deployment of language models, according to recent research findings?",
    "gold_answer_research": "Recent research findings suggest that common strategies for addressing the ethical and social risks associated with the deployment of language models include implementing alignment techniques to ensure that the models are helpful, honest, and harmless. These alignment techniques involve fine-tuning approaches such as reinforcement learning from human feedback to train the models to follow a broad class of written instructions. Additionally, it is important to consider the potential harms of language models in real-world applications and to evaluate how their outputs are used, especially in safety-critical situations. Furthermore, regulating access to large language models and incorporating diverse values-targeted datasets can help mitigate the risks of biased outputs, private data leaks, and misinformation generation.",
    "gold_answer_marketing": "Recent research findings suggest that common strategies for addressing ethical and social risks associated with the deployment of language models include mitigating biases, protecting private data, preventing the generation of misinformation, and ensuring alignment with human intentions to be helpful, honest, and harmless. These strategies aim to minimize the potential harms that language models may cause in various domains, such as medical diagnoses, employment, housing, and law enforcement."
  },
168: {"question": "What categories are the listed companies classified into, and what are some examples of tasks that can be performed in relation to text extraction, generation, rewriting, and chat services based on the given content?",
    "gold_answer_research": "The listed companies are classified as follows: Apple falls under the category of Technology, Facebook is categorized as Social Media, and Fedex's category is not specified. Some examples of tasks that can be performed in relation to text extraction include salient span masking, named entity identification, and adding an empty null document. For text generation, tasks can include natural language generation, speech recognition, and machine translation. For rewriting, tasks can involve summarization, paraphrasing, and producing rap lyrics based on a given article. Chat services can include customer assistance, complaint resolution, and information retrieval.",
    "gold_answer_marketing": "Apple is classified into the Technology category, Facebook is classified into the Social Media category, and Fedex is not classified. Tasks that can be performed in relation to text extraction include salient span masking and adding an empty null document. Tasks related to text generation include speech recognition, machine translation, and natural language generation. For rewriting, tasks can include summarization and rewriting rap lyrics. Chat services can involve customer assistance and complaints."
  },
171: {"question": "How does the use of retrieval-augmented generation methods contribute to improving reasoning or factual accuracy in large language models?",
  "gold_answer_research": "Retrieval augmented generation methods contribute to improving reasoning or factual accuracy in large language models by explicitly retrieving relevant passages from a large corpus and incorporating this information into the generation process. This enhances the factual grounding of outputs by providing up-to-date knowledge. By augmenting the input of LLMs with relevant passages, RAG methods reduce factual errors in knowledge-intensive tasks, improving the overall quality of generations and enhancing the versatility of LLMs. This approach combines the strengths of both factual-based and reasoning-oriented methods to address multi-step inference problems effectively.",
  "gold_answer_marketing": "Retrieval-augmented generation methods enhance the factual grounding of Large Language Models by explicitly retrieving relevant passages from a large corpus and incorporating this information into the generation process. This approach helps improve the reasoning and factual accuracy of the outputs by ensuring they are based on up-to-date knowledge and information."
  },
179: {"question": "Which prompt-based approach for agentic reasoning employs an interleaved sequence of Thought, Action, and Observation in its core mechanism, and is specifically mentioned as a foundational method to mitigate hallucination compared to Chain-of-Thought (CoT)?",
  "gold_answer_research": "The prompt-based approach for agentic reasoning that employs an interleaved sequence of Thought, Action, and Observation in its core mechanism, and is specifically mentioned as a foundational method to mitigate hallucination compared to Chain-of-Thought (CoT) is ReAct (Reason+Act) [Yao et al., 2023]. ReAct aims to create synergy between reasoning processes and action-taking capabilities within an LLM by prompting the model with examples that demonstrate this sequence. This method is designed to reduce hallucinations and improve decision-making transparency by incorporating external information via actions.",
  "gold_answer_marketing": "The prompt-based approach for agentic reasoning that employs an interleaved sequence of Thought, Action, and Observation in its core mechanism and is specifically mentioned as a foundational method to mitigate hallucination compared to Chain-of-Thought (CoT) is ReAct (Reason+Act) [Yao et al., 2023]."},
183: {"question": "How do language model pre-training tasks contribute to the effectiveness of open-domain question answering systems?",
  "gold_answer_research": "Language model pre-training tasks play a crucial role in enhancing the effectiveness of open-domain question answering systems. By leveraging tasks such as Next Sentence Prediction (NSP) and domain-specific pre-training, models like BERT can better handle noisy and irrelevant contexts, leading to improved performance. Additionally, augmenting queries with relevant contexts can significantly enhance the unsupervised machine reading capabilities of pre-trained language models, ultimately improving their ability to provide accurate and up-to-date information for dynamic knowledge-intensive tasks.",
  "gold_answer_marketing": "Language model pre-training tasks contribute to the effectiveness of open-domain question answering systems by providing a foundation of knowledge and context for the model to build upon. This allows the model to better understand and generate answers to a wide range of questions. By fine-tuning the pre-trained model on specific tasks or domains, it can perform even better in answering questions accurately."},
186: {"question": "What roles do classic and neural information retrieval systems play in locating relevant context for answering questions?",
  "gold_answer_research": "Classic information retrieval systems focus on retrieving relevant documents based on keyword matching and other traditional techniques, while neural information retrieval systems utilize dense representations and neural networks to capture semantic relationships between queries and documents. Classic systems excel at efficiently retrieving relevant information based on exact matches, while neural systems are adept at understanding complex relationships and context within documents, thereby improving accuracy in answering questions that require deeper comprehension. Both systems play essential roles in locating relevant context for answering questions, with classic systems offering efficiency and neural systems providing enhanced understanding and accuracy in retrieving pertinent information.",
  "gold_answer_marketing": "Classic information retrieval systems focus on utilizing term-based vectors and contextual references through multiple retrieval iterations, while neural information retrieval systems involve neural network architectures to learn dense representations of text for enhanced retrieval capability. Neural IR is a new category of methods that aim to locate relevant context more efficiently, but it is not always necessary to outperform classic systems in order to be effective in answering questions."},
189: {"question": "How can instruction prompting improve model alignment and efficiency compared to few-shot learning?",
  "gold_answer_research": "Instruction prompting can improve model alignment and efficiency compared to few-shot learning by providing task-specific representations that can be used for multiple downstream language tasks without any additional training. By encoding inputs together with task instructions, the model can generate more accurate outputs based on the prompt provided, leading to improved performance on held-out tasks. Additionally, fine-tuning language models with instructions can enhance model alignment by optimizing input word embeddings instead of fine-tuning, resulting in better efficiency and scalability in handling larger sequence lengths and more complex tasks.",
  "gold_answer_marketing": "Instruction prompting can improve model alignment and efficiency by providing task-specific representations without needing additional training. This approach uses input-output pairs to fine-tune a pretrained language model to generate outputs based on provided prompts, which enhances performance on various tasks. This method contrasts with few-shot learning, where models are optimized on a range of tasks but instruction prompting focuses on improving performance on specific tasks with minimal additional training."},
193: {"question": "How can language models be augmented via external tools or APIs to extend their capabilities?",
  "gold_answer_research": "To augment language models with external tools or APIs, models like TALM and Toolformer fine-tune a language model to learn to use external tool APIs. The dataset is expanded based on whether an added API call annotation can enhance model outputs. This approach improves the generalization of language models by integrating parameterized knowledge from language models with non-parameterized data from external knowledge bases, as seen in the research on retrieval-augmented language models. This method significantly enhances the capabilities of language models by leveraging external tools and APIs to expand their functionality and improve performance in specialized tasks.",
  "gold_answer_marketing": "By fine-tuning language models with external tool APIs, such as TALM and Toolformer, their capabilities can be extended. This process involves expanding the dataset to assess if adding new API call annotations improves the model's output quality. This approach enhances generalization and effectiveness of language models."},
194: {"question": "What are some advantages of using attention mechanisms in neural network architectures?",
  "gold_answer_research": "Some advantages of using attention mechanisms in neural network architectures include the ability to relate different positions within a sequence, allowing for the computation of representations of the same sequence. This can be very useful for tasks such as machine reading, abstractive summarization, or image description generation. Attention mechanisms can also help in reducing memory and storage usage, which is beneficial for large Transformer models trained with algorithms like Adam. Additionally, attention mechanisms can address the issue of fixed-length context vectors, enabling neural networks to remember longer sentences and improve overall performance in various tasks.",
  "gold_answer_marketing": "Using attention mechanisms in neural network architectures can help improve memory and storage efficiency. Additionally, attention mechanisms can enhance the model's ability to relate different positions within a sequence, making it useful for tasks such as machine reading and image description generation."},
195: {"question": "How does self-attention differ from traditional attention mechanisms in sequence modeling?",
  "gold_answer_research": "Self-attention differs from traditional attention mechanisms in sequence modeling by focusing on relating different positions within a single sequence to compute a representation of that same sequence. Traditional attention mechanisms often involve aligning and weighting different parts of the input sequence to output a context vector. In contrast, self-attention directly looks at the relationships between all positions in the sequence simultaneously through the Key, Value, and Query mechanism, allowing for more comprehensive modeling of dependencies within the sequence. Additionally, self-attention has shown to be more effective in tasks such as machine reading, abstractive summarization, and image description generation compared to traditional attention mechanisms.",
  "gold_answer_marketing": "Self-attention differs from traditional attention mechanisms by focusing on different positions within the same sequence to compute a representation of that sequence, allowing for more efficient processing of information. This is in contrast to traditional attention mechanisms that typically focus on aligning different sequences or inputs in a modeling task."},
200: {"question": "What mechanisms can be used to enable agents to break down complex tasks into smaller, manageable steps?",
  "gold_answer_research": "One mechanism that can be used to enable agents to break down complex tasks into smaller, manageable steps is task decomposition. By breaking down large tasks into smaller subgoals, agents can efficiently handle complex tasks. Another mechanism is self-reflection and refinement, where agents can reflect on past actions, learn from mistakes, and refine their approach for future steps, improving the quality of final results. Additionally, utilizing advanced configurations offered by external APIs and tools can help agents understand and utilize options like result filtering, sorting criteria, specifying search domains, or interacting with structured databases, ultimately supporting more targeted and efficient task completion aligned with task demands.",
  "gold_answer_marketing": "Agents can use subgoal and decomposition techniques to break down large tasks into smaller, more manageable steps. By reflecting on past actions, learning from mistakes, and refining their approach, agents can improve the quality of their final results."},
201: {"question": "How do autonomous systems incorporate short-term and long-term memory, and what types of external storage or retrieval methods are leveraged?",
  "gold_answer_research": "Autonomous systems incorporate short-term memory through in-context learning, allowing the model to learn and retain information for immediate use. Long-term memory is utilized to store and recall extensive information over extended periods, often by utilizing an external vector store for fast retrieval. External storage and retrieval methods such as memory banks, document indexes, or memory streams are leveraged to provide a comprehensive repository of experiences and information for the autonomous systems to access and utilize in decision-making processes.",
  "gold_answer_marketing": "Autonomous systems utilize short-term memory for in-context learning and long-term memory for retaining and recalling vast amounts of information. External storage methods like memory banks and retrieval models are often used for fast data retrieval and processing."},
206: {"question": "What are the primary differences between white-box and black-box adversarial attacks?",
  "gold_answer_research": "White-box attacks involve attackers having full access to the model architecture, weights, and training pipeline, allowing them to obtain gradient signals for adversarial attacks. On the other hand, black-box attacks assume attackers only have access to an API-like service without detailed information about the model. White-box attacks typically require knowledge of the model's internal workings, while black-box attacks rely on external input-output interactions with the model, making them more challenging to execute. Additionally, white-box attacks often result in nonsensical prompts and can be detected by examining perplexity, whereas black-box attacks may use token manipulation or jailbreak prompting to trigger model failures while retaining semantic meanings.",
  "gold_answer_marketing": "White-box attacks have full access to model details like weights and architecture, while black-box attacks only have access to input-output interface like an API. White-box attacks are more powerful but black-box attacks are based on limited information."},
213: {"question": "How can external knowledge sources be leveraged to reduce or detect hallucinations in generated content?",
  "gold_answer_research": "External knowledge sources can be leveraged to reduce or detect hallucinations in generated content by integrating external or in-context knowledge during reasoning. By retrieving structured sources like databases or web content, models can provide factual grounding and reduce hallucinations. Additionally, utilizing internal contexts like prior interactions or training examples can enhance contextual coherence and help bridge logical gaps, ultimately improving the quality of the generated content. These strategies, such as IAG and RA-DT, aim to make the model less reliant on irrelevant passages and more accurate in its outputs.",
  "gold_answer_marketing": "External knowledge sources can be leveraged to reduce or detect hallucinations in generated content by integrating structured data from databases or web content, providing factual grounding. This can help bridge logical gaps and enhance contextual coherence in the generated output."},
215: {"question": "What are the effects of fine-tuning on new knowledge with respect to a model’s tendency to hallucinate?",
  "gold_answer_research": "Fine-tuning LLMs on new knowledge can lead to slower learning of examples with new knowledge compared to examples consistent with pre-existing knowledge. However, once the new knowledge examples are learned, it can increase the model's tendency to hallucinate. Using retrieval to ground model generation helps reduce hallucination, while error rates are higher for rarer entities and facts mentioned later in generation tasks. Platforms like FAVABench provide benchmarks for evaluating fine-grained hallucination errors in model responses.",
  "gold_answer_marketing": "Fine-tuning on new knowledge may initially slow down the learning process in LLMs compared to consistent knowledge. However, once the new knowledge is learned, it can actually reduce the model's tendency to hallucinate."},
217: {"question": "What approaches exist for calibrating models to better handle questions where the correct answer is unknown or unanswerable?",
  "gold_answer_research": "There are several approaches for calibrating models to handle questions with unknown or unanswerable correct answers. One approach is to measure the model's output uncertainty when faced with such questions. Another method is to prompt the model to generate responses to unanswerable questions, which may trigger hallucinations. Benchmarks like TruthfulQA and SelfAware are used to evaluate how well models can generate truthful responses in such cases. Additionally, converting claims into questions, sampling multiple times from the model, and computing aggregated scores can help improve the model's handling of unknown or unanswerable questions.",
  "gold_answer_marketing": "Approaches for calibrating models to better handle questions where the correct answer is unknown or unanswerable include using the model's own confidence level as a proxy for truthfulness and implementing reference-free methods that rely on the model's output probabilities. These methods help ensure that the model accurately assesses its own uncertainty when facing such questions."},
220: {"question": "What challenges can arise from relying solely on automated or AI-based evaluation methods?",
  "gold_answer_research": "Relying solely on automated or AI-based evaluation methods can lead to challenges such as biases in the outputs, irrelevance, toxicity, or bias in the responses, disjointed or incoherent outputs when integrating retrieved information with different tasks, redundancy in responses when similar information is retrieved from multiple sources, and subjective preferences impacting the evaluation process. Additionally, automated evaluation systems may have noticeable biases and may not always provide accurate assessments of the AI applications. It is important to supplement automated evaluation with human evaluation to ensure comprehensive and reliable feedback on the product's performance.",
  "gold_answer_marketing": "Relying solely on automated or AI-based evaluation methods can lead to biased, irrelevant, or toxic outputs, which can lower the quality and reliability of responses. Without human evaluation, integration of retrieved information can be challenging, resulting in disjointed or repetitive outputs. Additionally, subjective preferences may play a significant role, impacting the accuracy and relevance of the evaluations."},
}



test_questions = {
10: {"question": "Which method incorporates relative position information by rotating the affine-transformed word embedding vector according to its position index, as described in the context of rotary position embedding?"},
12: {"question": "How are annotators instructed to determine which portions of a paper should be used to justify answers in the QASPER annotation process, and what considerations are involved when selecting evidence?"},
17: {"question": "How does LoRA manage to minimize inference time overhead, and what is the role of its low-rank update structure in enabling efficient parameter adaptation compared to traditional adapter-based methods?"},
24: {"question": "How is reward modeling applied to GPT-3 models to improve alignment with human preferences, and what are the key strategies used to collect and train on human comparison data for this technique?"},
37: {"question": "How does InstructGPT 175B respond differently from GPT-3 175B when given a prompt in Swedish, particularly regarding the use of language and adherence to instructions?"},
46: {"question": "How does retrieval augmentation enhance the capabilities of multimodal Transformer-based models in generating both text and images, and what architectural components are necessary to support handling sequences of multiple data types?"},
53: {"question": "How does the use of QLoRA with different quantization schemes impact the performance of LLaMA models across various dataset types and model sizes, particularly when evaluated on a comprehensive language understanding benchmark?"},
56: {"question": "How does the quantization process in large language models impact their ability to preserve model quality compared to alternative techniques, and what are some challenges or trade-offs associated with quantization methods such as SmoothQuant and LLM.int8()?"},
59: {"question": "What is the name of the algorithm that optimizes language models from human preferences using a simple binary cross-entropy objective, avoiding explicit reward modeling or reinforcement learning, and is compared to PPO-based RLHF in effectiveness?"},
61: {"question": "Which author derived the DPO objective, proved its theoretical properties, and contributed to the PPO and reward learning baselines?"},
65: {"question": "How does the DPO approach address user requests that involve privacy concerns or potentially illegal activities, and how does this compare to responses that provide only background information about the subject?"},
78: {"question": "How do annotation scores for informativeness, clarity, and readability vary depending on different types of questions?"},
86: {"question": "Which inference-time algorithm causes a large drop in PopQA and ASQA performance by always retrieving and using the top one document only, similar to standard RAG approaches?"},
114: {"question": "How does the performance of models trained with KTO compare to those aligned using SFT, DPO, and ORPO across different evaluation datasets, and what do the results suggest about the impact of different KTO variants and settings?"},
117: {"question": "Which model for multi-hop question answering emulates mammalian brain knowledge storage with KG triples and employs a personalized PageRank algorithm for retrieval?"},
120: {"question": "How do retrieval-augmented generation methods enhance question answering capabilities in contexts where both structured data, such as knowledge graphs or tables, and unstructured text are involved?"},
121: {"question": "Which benchmark provides 4,409 question-answer pairs, mock APIs to simulate web and Knowledge Graph search, and was the basis for a KDD Cup 2024 challenge?"},
158: {"question": "What is the definition of beamforming steering matrix and what key goal does it serve in relation to transmit antennas and signal-to-noise ratio (SNR)?"},
164: {"question": "Which framework remains limited by the lack of iterative feedback or dynamic retrieval, and is described as having issues with retrieval adequacy, reasoning depth, and system adaptability?"},
166: {"question": "Which approach in code generation tasks retrieves code documentation via OSCAT libraries to ensure syntactic accuracy and executable logic?"},
169: {"question": "How does retrieval-augmented generation leverage external information sources to enhance the reasoning abilities of large language models during complex question answering tasks?"},
173: {"question": "What are some of the unique retrieval and reasoning challenges that RAG-based systems encounter when applied to web-based tasks requiring agentic planning and dynamic tool use?"},
185: {"question": "How can answer extraction models be jointly trained with retrieval models, and what are the benefits of end-to-end approaches?"},
187: {"question": "In what ways can large pretrained language models be used for closed-book question answering?"},
197: {"question": "What is the significance of the key, value, and query concepts in transformer models?"},
205: {"question": "What are the main limitations or challenges faced by large language model-powered autonomous agents when executing real-world tasks?"},
212: {"question": "What are some common causes leading to unfaithful or fabricated outputs in large language models?"},
214: {"question": "Which types of evaluation metrics or benchmarks are used to assess factuality in language model outputs?"},
216: {"question": "How do different sampling strategies during text generation impact factual accuracy and hallucination rates?"},
225: {"question": "What actor plays the role of Thanos in the Marvel Universe and the role of Cable in Deadpool 2?"}
}


# 10-question subset — improved from 8q after topic cluster audit.
# Original 8 covered RAG core, attention/arch, fine-tuning, retrieval/IR.
# Added Q102 (evaluation cluster, 0 of 21 qs were represented) and
# Q213 (hallucination cluster, 0 of 9 qs were represented).
# Q212 was first choice but lives in test_questions, not validation.
# Q213 ("leveraging external knowledge to reduce hallucinations") is the
# closest hallucination question available in validation_questions_answers.
TUNING_SUBSET = [0, 4, 13, 15, 22, 102, 124, 134, 171, 213]


## 5. Shared Utilities

In [8]:
def format_docs(docs):
    return '\n\n'.join(d.page_content for d in docs)

def get_sources(docs):
    return [f"{d.metadata.get('doc_source','?')} (doc {d.metadata.get('doc_num','?')})" for d in docs]

def clean_answer(text):
    if '[/INST]' in text:
        text = text.split('[/INST]')[-1].strip()
    text = re.split(r'\bfor reference\b\s*[:,]?', text, flags=re.I)[0].strip()
    return text

engineering_template = '''[INST] You are a senior AI/ML engineer answering questions for a technical engineering team.
Provide a detailed, technically precise answer based ONLY on the context below.
Use only evidence present in the context. Do not add outside facts, prior knowledge, or guesses.
If the context does not contain enough information, explicitly state that and stop.
Include specific model names, methods, metrics, and technical details when available in context.

Context:
{context}

Question: {question}

Provide a thorough technical answer in 3-5 sentences. [/INST]'''

marketing_template = '''[INST] You are a helpful AI assistant answering questions for a marketing team.
Provide a clear, concise, and accessible answer based ONLY on the context below.
Avoid excessive jargon. Explain technical concepts simply so a non-engineer can understand.
Use only evidence present in the context. Do not add outside facts, prior knowledge, or guesses.
If the context does not contain enough information, explicitly state that and stop.

Context:
{context}

Question: {question}

Provide a clear, accessible answer in 2-3 sentences. [/INST]'''

eng_prompt = ChatPromptTemplate.from_template(engineering_template)
mkt_prompt = ChatPromptTemplate.from_template(marketing_template)

In [9]:
# ---- Metrics (same as NB1) ----
_tok_re = re.compile(r'[a-z0-9]+')
def normalize_tokens(text): return _tok_re.findall((text or '').lower())
def compute_rouge(p, r):
    s = rs.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True).score(r, p)
    return {'rouge1_f':s['rouge1'].fmeasure,'rouge2_f':s['rouge2'].fmeasure,'rougeL_f':s['rougeL'].fmeasure}
def compute_sem(p,r,e): return float(cos_sim([e.embed_query(p)],[e.embed_query(r)])[0][0])
def kw_recall(p,r):
    rw=set(normalize_tokens(r));pw=set(normalize_tokens(p))
    return len(rw&pw)/len(rw) if rw else 0.0
def ctx_overlap(p,c):
    pw=set(normalize_tokens(p));cw=set(normalize_tokens(c))
    return len(pw&cw)/len(pw) if pw else 0.0
def ctx_recall(c,r):
    rw=set(normalize_tokens(r));cw=set(normalize_tokens(c))
    return len(rw&cw)/len(rw) if rw else 0.0
def compute_all_metrics(p, r, e, c=''):
    ro=compute_rouge(p,r); ss=compute_sem(p,r,e); kw=kw_recall(p,r)
    co=ctx_overlap(p,c); cr=ctx_recall(c,r)
    comp = 0.25*ro['rouge1_f']+0.10*ro['rouge2_f']+0.10*ro['rougeL_f']+0.25*ss+0.15*kw+0.10*co+0.05*cr
    return {**ro,'semantic_similarity':ss,'keyword_recall':kw,
            'context_answer_overlap':co,'context_reference_recall':cr,'composite_score':comp}

eval_embeddings = HuggingFaceEmbeddings(model_name='multi-qa-mpnet-base-dot-v1')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

---
## 6. Technique A: Baseline (best config from NB1)

In [10]:
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
base_splits = splitter.split_documents(all_documents)
for i,s in enumerate(base_splits): s.metadata['split_id'] = i
base_emb = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
base_vs = QdrantVectorStore.from_documents(
    base_splits, base_emb, location=':memory:', collection_name='baseline', force_recreate=True
)
base_retriever = base_vs.as_retriever(search_type=RETRIEVER_TYPE, search_kwargs={'k': RETRIEVER_K})
print(f'Baseline: {len(base_splits)} chunks')

def make_chain(retriever, llm, prompt_tmpl):
    return ({'context': retriever | format_docs, 'question': RunnablePassthrough()}
            | prompt_tmpl | llm | output_parser)

def eval_pipeline(retriever, llm, label, qids=TUNING_SUBSET):
    rows = []
    for qid in qids:
        qa = validation_questions_answers[qid]
        for persona, gk, pt in [('eng','gold_answer_research',eng_prompt),('mkt','gold_answer_marketing',mkt_prompt)]:
            chain = make_chain(retriever, llm, pt)
            try:
                docs = retriever.invoke(qa['question'])
                ctx = format_docs(docs)
                ans = clean_answer(chain.invoke(qa['question']))
            except Exception as e:
                ans, ctx = f'ERROR: {e}', ''
            m = compute_all_metrics(ans, qa[gk], eval_embeddings, ctx) if not ans.startswith('ERROR') else \
                {k:0.0 for k in ['rouge1_f','rouge2_f','rougeL_f','semantic_similarity',
                 'keyword_recall','context_answer_overlap','context_reference_recall','composite_score']}
            rows.append({'technique':label, 'qid':qid, 'persona':persona, 'answer':ans, 'context':ctx[:300], **m})
        print(f'  [{label}] q{qid}')
    return pd.DataFrame(rows)

print('Running baseline...')
baseline_df = eval_pipeline(base_retriever, mistral_llm, 'baseline')
print(f'Baseline composite: {baseline_df["composite_score"].mean():.4f}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Baseline: 2540 chunks
Running baseline...


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [baseline] q0


...

  [baseline] q134


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [baseline] q171


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [baseline] q213
Baseline composite: 0.5797


---
## 7. Technique B: Query Expansion / HyDE

Generate a hypothetical answer using Mistral, embed that, then retrieve against it.
This bridges the semantic gap between short questions and long passages.

In [11]:
hyde_prompt_template = '''[INST] Write a short, detailed paragraph that would be a good answer to the following question.
Do not say "I don't know". Just write what a knowledgeable answer would look like.

Question: {question}

Answer: [/INST]'''

hyde_prompt = ChatPromptTemplate.from_template(hyde_prompt_template)
hyde_chain = hyde_prompt | mistral_llm | output_parser

def hyde_retriever_fn(question):
    """Generate hypothetical doc, embed it, retrieve against it."""
    hypo = clean_answer(hyde_chain.invoke({'question': question}))
    docs = base_vs.similarity_search(hypo, k=RETRIEVER_K)
    return docs

class HyDERetriever:
    def invoke(self, question):
        return hyde_retriever_fn(question)

hyde_ret = HyDERetriever()

def eval_custom_retriever(custom_ret, llm, label, qids=TUNING_SUBSET):
    rows = []
    for qid in qids:
        qa = validation_questions_answers[qid]
        for persona, gk, pt in [('eng','gold_answer_research',eng_prompt),('mkt','gold_answer_marketing',mkt_prompt)]:
            try:
                docs = custom_ret.invoke(qa['question'])
                ctx = format_docs(docs)
                prompt_filled = pt.format(context=ctx, question=qa['question'])
                ans = clean_answer(llm.invoke(prompt_filled))
            except Exception as e:
                ans, ctx = f'ERROR: {e}', ''
            m = compute_all_metrics(ans, qa[gk], eval_embeddings, ctx) if not ans.startswith('ERROR') else \
                {k:0.0 for k in ['rouge1_f','rouge2_f','rougeL_f','semantic_similarity',
                 'keyword_recall','context_answer_overlap','context_reference_recall','composite_score']}
            rows.append({'technique':label,'qid':qid,'persona':persona,'answer':ans,'context':ctx[:300],**m})
        print(f'  [{label}] q{qid}')
    return pd.DataFrame(rows)

print('Running HyDE...')
hyde_df = eval_custom_retriever(hyde_ret, mistral_llm, 'hyde')
print(f'HyDE composite: {hyde_df["composite_score"].mean():.4f}')

Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running HyDE...


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [hyde] q0


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  [hyde] q134


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [hyde] q171


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [hyde] q213
HyDE composite: 0.5739


---
## 8. Technique C: Parent Document Retrieval

Index **small chunks** (e.g. 256 chars) for precise retrieval, but when a small chunk
matches, return the **parent chunk** (e.g. 1500 chars) to the LLM for richer context.

In [12]:
child_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)

parent_emb = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
parent_vs = QdrantVectorStore.from_documents(
    [], parent_emb, location=':memory:', collection_name='parent_doc', force_recreate=True
)
docstore = InMemoryStore()

parent_retriever = ParentDocumentRetriever(
    vectorstore=parent_vs,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)
parent_retriever.add_documents(all_documents)
print(f'Parent doc retriever initialized. Docstore entries: {len(list(docstore.yield_keys()))}')

print('Running Parent Doc Retrieval...')
parent_df = eval_pipeline(parent_retriever, mistral_llm, 'parent_doc')
print(f'Parent Doc composite: {parent_df["composite_score"].mean():.4f}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Parent doc retriever initialized. Docstore entries: 2425
Running Parent Doc Retrieval...


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [parent_doc] q0


...

  [parent_doc] q134


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [parent_doc] q171


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [parent_doc] q213
Parent Doc composite: 0.5620


---
## 9. Technique D: Cross-Encoder Reranker

Retrieve a broad set (k=12) with dense retrieval, then use a cross-encoder to
rerank and keep only the top 4 most relevant passages.

In [13]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

broad_retriever = base_vs.as_retriever(search_type='similarity', search_kwargs={'k': 12})

class RerankedRetriever:
    def __init__(self, base_retriever, cross_encoder, top_k=4):
        self.base = base_retriever
        self.ce = cross_encoder
        self.top_k = top_k

    def invoke(self, question):
        docs = self.base.invoke(question)
        if not docs:
            return docs
        pairs = [(question, d.page_content) for d in docs]
        scores = self.ce.predict(pairs)
        ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
        return [doc for _, doc in ranked[:self.top_k]]

reranked_ret = RerankedRetriever(broad_retriever, cross_encoder, top_k=4)

print('Running Reranker...')
rerank_df = eval_custom_retriever(reranked_ret, mistral_llm, 'reranker')
print(f'Reranker composite: {rerank_df["composite_score"].mean():.4f}')

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

  [reranker] q134


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [reranker] q171


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [reranker] q213
Reranker composite: 0.5658


---
## 10. Compare All Techniques

In [14]:
all_techniques = pd.concat([baseline_df, hyde_df, parent_df, rerank_df])

print('=== TECHNIQUE COMPARISON (Mistral, 10-question subset) ===')
summary = all_techniques.groupby('technique')[[
    'composite_score','semantic_similarity','rouge1_f','keyword_recall',
    'context_answer_overlap','context_reference_recall'
]].mean().sort_values('composite_score', ascending=False)
print(summary.to_string())

print('\nBy persona (first-run showed engineering underperforms marketing by ~0.017):')
persona_table = all_techniques.groupby(['technique','persona'])[['composite_score']].mean().unstack()
print(persona_table.to_string())

eng_gap = all_techniques.groupby(['technique','persona'])['composite_score'].mean().unstack()
if 'engineering' in eng_gap.columns and 'marketing' in eng_gap.columns:
    eng_gap['eng_mkt_delta'] = eng_gap['engineering'] - eng_gap['marketing']
    print('\nEngineering - Marketing delta (negative = engineering worse):')
    print(eng_gap[['eng_mkt_delta']].to_string())

q13_scores = all_techniques[all_techniques['qid'] == 13]
if not q13_scores.empty:
    print('\nQ13 (Longformer outlier) scores by technique:')
    print(q13_scores.groupby('technique')[['composite_score']].mean().sort_values('composite_score', ascending=False).to_string())

best_technique = summary.index[0]
print(f'\n>>> Best technique: {best_technique}')

=== TECHNIQUE COMPARISON (Mistral, 10-question subset) ===
            composite_score  semantic_similarity  rouge1_f  keyword_recall  context_answer_overlap  context_reference_recall
technique                                                                                                                   
baseline           0.579680             0.832407  0.496400        0.486589                0.751889                  0.759131
hyde               0.573873             0.836912  0.474907        0.473049                0.804515                  0.771736
reranker           0.565767             0.837162  0.479597        0.465972                0.718159                  0.679155
parent_doc         0.561978             0.836433  0.489205        0.496957                0.663004                  0.655314

By persona (first-run showed engineering underperforms marketing by ~0.017):
           composite_score          
persona                eng       mkt
technique                           
ba

## 11. Technique Combinations (Mistral, 0 Cohere calls)

Individual techniques may stack. Test the most promising combinations before spending Cohere calls.

In [15]:
# Test combinations of the top techniques from Section 10.
# Only combine techniques that won or placed 2nd individually.

combo_results = []

# Combo 1: HyDE + Reranker — expand query AND rerank results
print('--- hyde_reranker ---')
class HyDERerankedRetriever:
    def __init__(self, hyde_chain, base_vs, cross_enc, broad_k=12, top_k=8):
        self.hyde_chain = hyde_chain
        self.base_vs = base_vs
        self.cross_enc = cross_enc
        self.broad_k = broad_k
        self.top_k = top_k
    def invoke(self, question):
        hypo = clean_answer(self.hyde_chain.invoke({'question': question}))
        docs = self.base_vs.similarity_search(hypo, k=self.broad_k)
        if len(docs) <= self.top_k:
            return docs
        pairs = [(question, d.page_content) for d in docs]
        scores = self.cross_enc.predict(pairs)
        ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
        return [d for _, d in ranked[:self.top_k]]

hyde_rerank_ret = HyDERerankedRetriever(hyde_chain, base_vs, cross_encoder, broad_k=12, top_k=RETRIEVER_K)
hyde_rerank_df = eval_custom_retriever(hyde_rerank_ret, mistral_llm, 'hyde_reranker')
combo_results.append(hyde_rerank_df)

# Combo 2: Parent Doc + Reranker — precise child matching, reranked parents
print('\n--- parent_reranker ---')
class ParentRerankedRetriever:
    def __init__(self, parent_ret, cross_enc, broad_k=12, top_k=8):
        self.parent_ret = parent_ret
        self.cross_enc = cross_enc
        self.broad_k = broad_k
        self.top_k = top_k
    def invoke(self, question):
        docs = self.parent_ret.invoke(question)
        if len(docs) <= self.top_k:
            return docs
        pairs = [(question, d.page_content) for d in docs]
        scores = self.cross_enc.predict(pairs)
        ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
        return [d for _, d in ranked[:self.top_k]]

parent_rerank_ret = ParentRerankedRetriever(parent_retriever, cross_encoder, broad_k=12, top_k=RETRIEVER_K)
parent_rerank_df = eval_custom_retriever(parent_rerank_ret, mistral_llm, 'parent_reranker')
combo_results.append(parent_rerank_df)

combo_df = pd.concat(combo_results)
all_with_combos = pd.concat([all_techniques, combo_df])

print('\n=== ALL TECHNIQUES + COMBINATIONS (Mistral, 10-question subset) ===')
full_summary = all_with_combos.groupby('technique')[[
    'composite_score','semantic_similarity','rouge1_f','keyword_recall',
    'context_answer_overlap','context_reference_recall'
]].mean().sort_values('composite_score', ascending=False)
print(full_summary.to_string())

print('\nBy persona:')
print(all_with_combos.groupby(['technique','persona'])[['composite_score']].mean().unstack().to_string())

q13_all = all_with_combos[all_with_combos['qid'] == 13]
if not q13_all.empty:
    print('\nQ13 (Longformer outlier) — did combinations fix it?')
    print(q13_all.groupby('technique')[['composite_score']].mean().sort_values('composite_score', ascending=False).to_string())

best_overall = full_summary.index[0]
print(f'\n>>> Best technique (including combos): {best_overall}')

Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- hyde_reranker ---


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [hyde_reranker] q0


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  [parent_reranker] q134


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [parent_reranker] q171


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [parent_reranker] q213

=== ALL TECHNIQUES + COMBINATIONS (Mistral, 10-question subset) ===
                 composite_score  semantic_similarity  rouge1_f  keyword_recall  context_answer_overlap  context_reference_recall
technique                                                                                                                        
baseline                0.579680             0.832407  0.496400        0.486589                0.751889                  0.759131
hyde                    0.573873             0.836912  0.474907        0.473049                0.804515                  0.771736
hyde_reranker           0.569466             0.834158  0.466747        0.461773                0.819871                  0.791024
parent_reranker         0.568790             0.839489  0.500755        0.501351                0.661436                  0.655314
reranker                0.565767             0.837162  0.479597        0.465972                0.718159                  0.679

## 12. G-Eval Dry Run with Mistral (0 Cohere calls)

Validate the judge prompt format and score parsing work correctly before spending Cohere calls.
Mistral's scores won't be as reliable as Cohere's, but they verify the pipeline works.

In [16]:
# G-Eval judge prompt and scoring — shared by both Mistral dry run and Cohere G-Eval.

judge_template = '''You are an expert evaluator for a question-answering system.

Given a QUESTION, a GOLD ANSWER (the ideal answer), and a CANDIDATE ANSWER (from the system),
rate the candidate answer on a scale of 1-5 for each criterion:

1. **Accuracy** (1-5): How factually correct is the candidate compared to the gold answer?
2. **Completeness** (1-5): Does the candidate cover all key points from the gold answer?
3. **Relevance** (1-5): Does the candidate stay on topic and answer the actual question?
4. **Tone** (1-5): Is the tone appropriate for the target audience ({persona})?

QUESTION: {question}

GOLD ANSWER: {gold_answer}

CANDIDATE ANSWER: {candidate_answer}

Respond with ONLY a JSON object in this exact format (no other text):
{{"accuracy": <int>, "completeness": <int>, "relevance": <int>, "tone": <int>}}'''

judge_prompt = ChatPromptTemplate.from_template(judge_template)

def parse_judge_scores(response_text):
    """Extract JSON scores from judge response, handling common formatting issues."""
    match = re.search(r'\{[^}]+\}', response_text)
    if match:
        try:
            scores = json.loads(match.group())
            return {k: int(v) for k, v in scores.items() if k in ('accuracy','completeness','relevance','tone')}
        except (json.JSONDecodeError, ValueError):
            pass
    return {'accuracy': 0, 'completeness': 0, 'relevance': 0, 'tone': 0}

# --- Mistral dry run ---
mistral_judge_chain = judge_prompt | mistral_llm | output_parser

def run_geval_mistral(technique_df, sample_size=3):
    """G-Eval with Mistral. Scores are less reliable than Cohere but free."""
    sample = technique_df.sample(n=min(sample_size, len(technique_df)), random_state=42)
    judge_results = []
    for _, row in sample.iterrows():
        qid = row['qid']
        persona = row['persona']
        qa = validation_questions_answers[qid]
        gk = 'gold_answer_research' if persona == 'eng' else 'gold_answer_marketing'
        persona_label = 'a technical engineering team' if persona == 'eng' else 'a non-technical marketing team'
        try:
            resp = mistral_judge_chain.invoke({
                'question': qa['question'],
                'gold_answer': qa[gk],
                'candidate_answer': row['answer'],
                'persona': persona_label
            })
            scores = parse_judge_scores(resp)
        except Exception as e:
            print(f'  Mistral judge error q{qid}/{persona}: {e}')
            scores = {'accuracy':0,'completeness':0,'relevance':0,'tone':0}
        scores['qid'] = qid
        scores['persona'] = persona
        scores['technique'] = row['technique']
        scores['avg_score'] = np.mean([scores['accuracy'],scores['completeness'],scores['relevance'],scores['tone']])
        judge_results.append(scores)
        print(f'  [Mistral] Judged q{qid}/{persona}: acc={scores["accuracy"]} comp={scores["completeness"]} rel={scores["relevance"]} tone={scores["tone"]}')
    return pd.DataFrame(judge_results)

# Run on baseline and best technique with Mistral (3 samples each = 6 local calls, 0 Cohere)
print('=== G-EVAL DRY RUN (Mistral as judge, 0 Cohere calls) ===')
print('\nBaseline:')
baseline_geval_m = run_geval_mistral(baseline_df, sample_size=3)
print(f'\n{best_overall}:')
best_tech_rows = all_with_combos[all_with_combos['technique'] == best_overall]
best_geval_m = run_geval_mistral(best_tech_rows, sample_size=3)

geval_m_all = pd.concat([baseline_geval_m, best_geval_m])
print('\nMistral G-Eval summary:')
print(geval_m_all.groupby('technique')[['accuracy','completeness','relevance','tone','avg_score']].mean().to_string())

json_success = (geval_m_all[['accuracy','completeness','relevance','tone']] > 0).all(axis=1).mean()
print(f'\nJSON parse success rate: {json_success:.0%}')
if json_success < 0.5:
    print('WARNING: Mistral struggles with JSON output. Consider simplifying the judge prompt for Mistral.')
else:
    print('Judge prompt works — safe to proceed with Cohere G-Eval.')

Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== G-EVAL DRY RUN (Mistral as judge, 0 Cohere calls) ===

Baseline:


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [Mistral] Judged q0/eng: acc=0 comp=0 rel=0 tone=0


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  [Mistral] Judged q134/mkt: acc=0 comp=0 rel=0 tone=0

baseline:


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [Mistral] Judged q0/eng: acc=0 comp=0 rel=0 tone=0


Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [Mistral] Judged q171/mkt: acc=0 comp=0 rel=0 tone=0
  [Mistral] Judged q134/mkt: acc=0 comp=0 rel=0 tone=0

Mistral G-Eval summary:
           accuracy  completeness  relevance  tone  avg_score
technique                                                    
baseline        0.0           0.0        0.0   0.0        0.0

JSON parse success rate: 0%


---
## 11. LLM-as-Judge / G-Eval (Reference-Based, Cohere)

Use Cohere as a judge to score RAG answers 1-5 against gold answers.
This gives a human-aligned quality signal that automated metrics miss.

**Why not RAGAS?** RAGAS requires an OpenAI key by default and is designed as a
full framework. The assignment allows GPT only if using RAGAS specifically.
Since we want to keep things simple and within Cohere quota, we implement
a lightweight G-Eval style judge directly with Cohere.
This also demonstrates deeper understanding than just calling a library.

In [17]:
# Cohere G-Eval — uses judge_prompt and parse_judge_scores defined in the dry run cell above.
judge_chain = judge_prompt | cohere_llm | output_parser

def run_geval(technique_df, sample_size=5):
    """Run G-Eval on a sample of results. Each call uses 1 Cohere API call."""
    sample = technique_df.sample(n=min(sample_size, len(technique_df)), random_state=42)
    judge_results = []
    for _, row in sample.iterrows():
        qid = row['qid']
        persona = row['persona']
        qa = validation_questions_answers[qid]
        gk = 'gold_answer_research' if persona == 'eng' else 'gold_answer_marketing'
        persona_label = 'a technical engineering team' if persona == 'eng' else 'a non-technical marketing team'
        try:
            resp = judge_chain.invoke({
                'question': qa['question'],
                'gold_answer': qa[gk],
                'candidate_answer': row['answer'],
                'persona': persona_label
            })
            scores = parse_judge_scores(resp)
            time.sleep(1.5)
        except Exception as e:
            print(f'  Judge error q{qid}/{persona}: {e}')
            scores = {'accuracy':0,'completeness':0,'relevance':0,'tone':0}
        scores['qid'] = qid
        scores['persona'] = persona
        scores['technique'] = row['technique']
        scores['avg_score'] = np.mean([scores['accuracy'],scores['completeness'],scores['relevance'],scores['tone']])
        judge_results.append(scores)
        print(f'  Judged q{qid}/{persona}: {scores}')
    return pd.DataFrame(judge_results)

print('Cohere API calls needed: ~5 per technique sample')

Cohere API calls needed: ~5 per technique sample


In [18]:
# Run G-Eval on baseline and best technique
print('G-Eval on baseline...')
baseline_geval = run_geval(baseline_df, sample_size=5)

# Run on the best technique from the comparison
best_tech_df = all_with_combos[all_with_combos['technique'] == best_overall]
print(f'\nG-Eval on {best_overall}...')
best_geval = run_geval(best_tech_df, sample_size=5)

geval_all = pd.concat([baseline_geval, best_geval])
print('\n=== G-EVAL RESULTS ===')
print(geval_all.groupby('technique')[['accuracy','completeness','relevance','tone','avg_score']].mean().to_string())

G-Eval on baseline...
  Judged q0/eng: {'accuracy': 5, 'completeness': 3, 'relevance': 5, 'tone': 5, 'qid': 0, 'persona': 'eng', 'technique': 'baseline', 'avg_score': np.float64(4.5)}
  Judged q171/mkt: {'accuracy': 4, 'completeness': 3, 'relevance': 5, 'tone': 5, 'qid': 171, 'persona': 'mkt', 'technique': 'baseline', 'avg_score': np.float64(4.25)}
  Judged q134/mkt: {'accuracy': 5, 'completeness': 4, 'relevance': 5, 'tone': 5, 'qid': 134, 'persona': 'mkt', 'technique': 'baseline', 'avg_score': np.float64(4.75)}
  Judged q0/mkt: {'accuracy': 5, 'completeness': 5, 'relevance': 5, 'tone': 5, 'qid': 0, 'persona': 'mkt', 'technique': 'baseline', 'avg_score': np.float64(5.0)}
  Judged q22/eng: {'accuracy': 5, 'completeness': 4, 'relevance': 5, 'tone': 5, 'qid': 22, 'persona': 'eng', 'technique': 'baseline', 'avg_score': np.float64(4.75)}

G-Eval on baseline...
  Judged q0/eng: {'accuracy': 5, 'completeness': 3, 'relevance': 5, 'tone': 5, 'qid': 0, 'persona': 'eng', 'technique': 'baseline', 

---
## 12. Final Full Run (75 questions, Cohere, best technique)

Use the best-performing technique + Cohere for the required full 75-question evaluation.

In [19]:
# Build the best retriever based on winning technique
# Adjust this cell based on which technique won in section 10.
# Example: if reranker won, use reranked_ret; if baseline, use base_retriever

BEST_RETRIEVER = base_retriever  # <-- CHANGE based on section 10 results
BEST_LABEL = 'baseline_cohere'   # <-- CHANGE label accordingly

print(f'Full 75-question run with {BEST_LABEL}...')
full_results = []
all_qids = list(validation_questions_answers.keys())

for qid in all_qids:
    qa = validation_questions_answers[qid]
    for persona, gk, pt in [('engineering','gold_answer_research',eng_prompt),
                             ('marketing','gold_answer_marketing',mkt_prompt)]:
        chain = make_chain(BEST_RETRIEVER, cohere_llm, pt)
        try:
            docs = BEST_RETRIEVER.invoke(qa['question'])
            ctx = format_docs(docs)
            ans = clean_answer(chain.invoke(qa['question']))
            time.sleep(1.2)
        except Exception as e:
            ans, ctx = f'ERROR: {e}', ''
        m = compute_all_metrics(ans, qa[gk], eval_embeddings, ctx) if not ans.startswith('ERROR') else \
            {k:0.0 for k in ['rouge1_f','rouge2_f','rougeL_f','semantic_similarity',
             'keyword_recall','context_answer_overlap','context_reference_recall','composite_score']}
        full_results.append({'qid':qid,'persona':persona,'question':qa['question'],
                             'answer':ans,'gold':qa[gk],'context':ctx[:500],**m})
    print(f'  q{qid}')

full_df = pd.DataFrame(full_results)
cols = ['rouge1_f','rouge2_f','rougeL_f','semantic_similarity','keyword_recall',
        'context_answer_overlap','context_reference_recall','composite_score']

print(f'\nOverall composite: {full_df["composite_score"].mean():.4f}')
print('\nBy persona:')
print(full_df.groupby('persona')[cols].mean().to_string())

print('\nPer-question sample (first 10):')
print(full_df[['qid','persona','composite_score','semantic_similarity','rouge1_f']].head(10).to_string(index=False))

Full 75-question run with baseline_cohere...
  q0
  q4
  q5
  q11
  q13
  q15
  q16
  q20
  q22
  q23
  q27
  q28
  q33
  q36
  q38
  q41
  q44
  q54
  q58
  q72
  q73
  q80
  q83
  q90
  q91
  q93
  q94
  q95
  q102
  q103
  q108
  q111
  q112
  q118
  q119
  q123
  q124
  q126
  q128
  q129
  q130
  q133
  q134
  q135
  q136
  q139
  q140
  q143
  q145
  q146
  q150
  q151
  q154
  q155
  q157
  q160
  q161
  q162
  q165
  q168
  q171
  q179
  q183
  q186
  q189
  q193
  q194
  q195
  q200
  q201
  q206
  q213
  q215
  q217
  q220

Overall composite: 0.0846

By persona:
             rouge1_f  rouge2_f  rougeL_f  semantic_similarity  keyword_recall  context_answer_overlap  context_reference_recall  composite_score
persona                                                                                                                                          
engineering  0.063734  0.024999  0.039040             0.127492        0.078658                0.120735                  0.112568 

---
## 13. G-Eval on Full Run Sample

In [20]:
# Run G-Eval on a sample of the full run (10 samples = 10 Cohere calls)
print('G-Eval on full run sample...')
# Rename persona for compatibility
full_df_for_geval = full_df.copy()
full_df_for_geval['persona'] = full_df_for_geval['persona'].map({'engineering':'eng','marketing':'mkt'})
full_df_for_geval['technique'] = BEST_LABEL

full_geval = run_geval(full_df_for_geval, sample_size=10)
print('\n=== FULL RUN G-EVAL ===')
print(full_geval.groupby('persona')[['accuracy','completeness','relevance','tone','avg_score']].mean().to_string())
print(f'\nOverall avg G-Eval score: {full_geval["avg_score"].mean():.2f}/5.0')

G-Eval on full run sample...
  Judge error q124/mkt: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-control': 'no-cache, no-store, no-transform, must-revalidate, private, max-age=0', 'content-encoding': 'gzip', 'content-type': 'application/json', 'expires': 'Thu, 01 Jan 1970 00:00:00 GMT', 'pragma': 'no-cache', 'vary': 'Origin,Accept-Encoding', 'x-accel-expires': '0', 'x-debug-trace-id': '8a7ceb5c929ad05f0bc102f648a2289c', 'date': 'Fri, 10 Apr 2026 05:52:44 GMT', 'x-envoy-upstream-service-time': '4', 'server': 'envoy', 'via': '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000', 'transfer-encoding': 'chunked'}, status_code: 429, body: {'id': 'b8b5ec3e-d9ac-4080-afc1-e2c72de9b580', 'message': "You are using a Trial key, which is limited to 20 API calls / minute. You can continue to use the Trial key for free or upgrade to a Production key with higher rate limits at 'https://dashboard.cohere.com/api-keys'. Contact us on 'https://discord.gg/XW44jPfYJu' or email us at su

---
## 14. Summary & Export

In [21]:
print('='*80)
print('EXPERIMENT SUMMARY')
print('='*80)
print(f'\nBase config from NB3: chunk={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}, '
      f'embed={EMBED_MODEL}, k={RETRIEVER_K}, type={RETRIEVER_TYPE}')
print(f'\nTechnique comparison (Mistral, {len(TUNING_SUBSET)}-question subset):')
print(full_summary.to_string())
print(f'\nBest technique (including combos): {best_overall}')
print(f'\nFull 75-question run composite ({BEST_LABEL}): {full_df["composite_score"].mean():.4f}')
print(f'G-Eval avg: {full_geval["avg_score"].mean():.2f}/5.0')

full_df.to_csv('full_75_results.csv', index=False)
print('\nSaved full_75_results.csv')

# Section 5.2: Test question outputs (Q0, Q134, Q225)
print('\n' + '='*80)
print('SECTION 5.2 — TEST QUESTION OUTPUTS')
print('='*80)

test_qids = [0, 134, 225]
for tqid in test_qids:
    if tqid in validation_questions_answers:
        qa = validation_questions_answers[tqid]
    elif tqid in test_questions:
        qa = test_questions[tqid]
    else:
        print(f'\nQ{tqid}: NOT FOUND in validation or test data')
        continue

    q = qa['question']
    print(f'\n{"="*60}')
    print(f'Q{tqid}: {q}')
    print(f'{"="*60}')

    if hasattr(BEST_RETRIEVER, 'invoke'):
        docs = BEST_RETRIEVER.invoke(q)
    else:
        docs = BEST_RETRIEVER.get_relevant_documents(q)

    for persona, pt in [('Engineering', eng_prompt), ('Marketing', mkt_prompt)]:
        chain = pt | cohere_llm | output_parser
        context = '\n'.join([d.page_content for d in docs])
        answer = clean_answer(chain.invoke({'context': context, 'question': q}))
        print(f'\n[{persona}]')
        print(answer)
        time.sleep(1.5)

EXPERIMENT SUMMARY

Base config from NB3: chunk=1500, overlap=300, embed=multi-qa-mpnet-base-dot-v1, k=8, type=similarity

Technique comparison (Mistral, 10-question subset):
                 composite_score  semantic_similarity  rouge1_f  keyword_recall  context_answer_overlap  context_reference_recall
technique                                                                                                                        
baseline                0.579680             0.832407  0.496400        0.486589                0.751889                  0.759131
hyde                    0.573873             0.836912  0.474907        0.473049                0.804515                  0.771736
hyde_reranker           0.569466             0.834158  0.466747        0.461773                0.819871                  0.791024
parent_reranker         0.568790             0.839489  0.500755        0.501351                0.661436                  0.655314
reranker                0.565767             

TooManyRequestsError: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-control': 'no-cache, no-store, no-transform, must-revalidate, private, max-age=0', 'content-encoding': 'gzip', 'content-type': 'application/json', 'expires': 'Thu, 01 Jan 1970 00:00:00 GMT', 'pragma': 'no-cache', 'vary': 'Origin,Accept-Encoding', 'x-accel-expires': '0', 'x-debug-trace-id': '300795744fd247d874895519dbc8e131', 'x-trial-endpoint-call-limit': '20', 'x-trial-endpoint-call-remaining': '2', 'date': 'Fri, 10 Apr 2026 05:53:16 GMT', 'x-envoy-upstream-service-time': '7', 'server': 'envoy', 'via': '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000', 'transfer-encoding': 'chunked'}, status_code: 429, body: {'id': '58124a63-09e0-4f0a-8e7b-97bb4ad8371c', 'message': "You are using a Trial key, which is limited to 1000 API calls / month. You can continue to use the Trial key for free or upgrade to a Production key with higher rate limits at 'https://dashboard.cohere.com/api-keys'. Contact us on 'h